# Business Entity Resolution: full pipeline

Runs the ML Challenge 2026 Business Entity Resolution pipeline end to end: normalize, blocking,
features, GBDT, Laya fine-tuning, ensemble, test-set inference, validate, package.

**Self-contained:** Section 0.2 writes every pipeline source file (`src/*.py`,
`requirements.txt`, `README.md`, the challenge's `validate_submission.py` and
`Documentation_template.md`) from `%%writefile` cells, so all the code is visible here. Nothing is
cloned or pulled at runtime. To change the pipeline, edit the `%%writefile` cell, re-run it, then
**restart the kernel** before re-running later steps so the new code is loaded.

**Hardware:** Sections 1-4 are CPU work (text processing and gradient boosting; a GPU doesn't
speed them up). Section 5 (Laya fine-tuning) and the Laya scoring in Sections 6-7 use the GPU.
On Kaggle choose **GPU T4 x2**: fine-tuning runs DDP across both GPUs, and scoring runs one
model copy per GPU in parallel.

Run cells **top to bottom**; each step reads what the previous one wrote. Every step checks its
inputs and names the earlier step to re-run if something is missing.


## 0. Setup

### 0.1 Check the GPU


In [ ]:
import subprocess

try:
    result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
    print(result.stdout or result.stderr)
except FileNotFoundError:
    print("nvidia-smi not found -- no GPU accelerator is selected for this session.")
    print("Kaggle: Settings (right sidebar) -> Accelerator -> GPU T4 x2, then re-run this cell.")
    print("Sections 1-4 run on CPU; only Sections 5-7 need the GPU, so you can add it before Section 5.")


### 0.2 Write the pipeline code

Creates the submission-package layout and writes every source file into it. Re-running these
cells overwrites the files with the version shown here.


In [ ]:
import os

# /kaggle/working on Kaggle, /content on Colab, else the notebook's own directory.
if os.path.isdir("/kaggle/working"):
    WORKDIR = "/kaggle/working"
elif os.path.isdir("/content"):
    WORKDIR = "/content"
else:
    WORKDIR = os.getcwd()

STUDENT_RESOURCE_DIR = f"{WORKDIR}/azlaya/student_resource"
PIPELINE_DIR = f"{STUDENT_RESOURCE_DIR}/code/business_entity_resolution"
for d in (f"{PIPELINE_DIR}/src", f"{STUDENT_RESOURCE_DIR}/utils"):
    os.makedirs(d, exist_ok=True)
print("STUDENT_RESOURCE_DIR:", STUDENT_RESOURCE_DIR)
print("PIPELINE_DIR:        ", PIPELINE_DIR)


In [ ]:
%%writefile $PIPELINE_DIR/requirements.txt
# Business Entity Resolution pipeline -- pinned for the GPU machine (Colab / AWS / Kaggle).
# Install with: pip install -r requirements.txt
# Python 3.10+ required (laya's own floor: huggingface_hub 1.x / transformers 5.x / torch 2.14
# all require it -- see NandhaKishorM/laya's README "Installation details").

# --- normalize.py / blocking.py / features.py / train_gbdt.py ---
#
# No upper bound on pandas or numpy: Kaggle/Colab base images ship pandas pre-built against
# whatever numpy (and, transitively, whatever pandas major version) they currently preinstall.
# An upper-bound pin here that's stricter than what's already installed makes pip downgrade one
# without rebuilding the other against it, which breaks pandas' compiled extension outright
# ("numpy.dtype size changed, may indicate binary incompatibility" on import) -- hit in practice
# on a live Kaggle run with a numpy<2.0 pin that used to be here. Only floor versions below.
pandas>=2.0
numpy>=1.24
pyarrow>=14.0            # parquet IO for the features_{split}.parquet tables
scikit-learn>=1.3        # TfidfVectorizer (blocking/features), LogisticRegression (ensemble)
scipy>=1.10              # sparse matrices for blocking
sparse_dot_topn>=1.1     # Apache-2.0; multithreaded sparse top-n matrix product (blocking at 10M+ records)
rapidfuzz>=3.6,<4.0      # Levenshtein / Jaro-Winkler; process.cpdist (vectorized pairwise) needs >=3.6
tqdm>=4.66               # progress bars in every long-running step
lightgbm>=4.1,<5.0
joblib>=1.3              # persists the logistic-regression stacker

# --- laya_finetune.py / ensemble.py / predict.py ---
#
# torch is intentionally NOT pinned here beyond what `laya` itself already requires
# (torch>=2.0.0, transformers>=4.48.0, safetensors>=0.4.0, huggingface_hub>=0.20.0 -- see laya's
# own pyproject.toml). Kaggle/Colab GPU runtimes ship torch pre-built against a specific matched
# CUDA driver; an independent `torch>=X.Y` pin here that's stricter than what's preinstalled can
# make pip replace that CUDA-matched build with a mismatched wheel from PyPI, silently breaking
# GPU access (same class of bug as the numpy/pandas one above, but worse -- it costs GPU quota to
# even notice). If `pip install -r requirements.txt` ever reinstalls torch, immediately check
# `python -c "import torch; print(torch.cuda.is_available())"` and reinstall from
# https://pytorch.org/get-started/locally/ for your exact CUDA version if it comes back False.
transformers>=4.48.0
safetensors>=0.4.0
huggingface_hub>=0.24.0
laya>=0.3.20,<0.4.0           # convaiinnovations/laya -- Apache-2.0, 421M/322M params (<=8B, license-compliant); pulls in a compatible torch as its own dependency if none is already installed

# Optional: train_gbdt.py is written for LightGBM (as specified). XGBoost is a documented
# drop-in alternative (same feature table / label / split logic) if you'd rather use it instead:
# xgboost>=2.0,<3.0


In [ ]:
%%writefile $PIPELINE_DIR/README.md
# Business Entity Resolution -- pipeline

Blocking + GBDT + fine-tuned Laya, stacked, for the ML Challenge 2026 Business Entity Resolution
task. Produces `output/candidate_pairs.tsv` and `output/matching_results.tsv`.

## Easiest way to run: the notebook

`notebooks/run_pipeline_colab_kaggle.ipynb` is self-contained: it writes every file in `src/`
(plus this README, `requirements.txt`, the challenge's `validate_submission.py` and
`Documentation_template.md`) from `%%writefile` cells, then runs each step in-kernel with live
progress bars. Upload it to Kaggle, attach the dataset, choose **GPU T4 x2**, and run top to bottom.
Nothing is cloned at runtime.

The notebook is generated from `src/` by `notebooks/build_notebook.py`. After editing anything in
`src/`, regenerate it from this directory with `python notebooks/build_notebook.py`.

## Layout

```
student_resource/
├── dataset/{train,test}/...
├── utils/validate_submission.py
├── output/                  <- candidate_pairs.tsv, matching_results.tsv (predict.py)
├── data_processed/          <- normalized TSVs, candidate/feature tables, Laya datasets, reports
├── models/                  <- GBDT, fine-tuned Laya checkpoints, ensemble stackers
└── code/business_entity_resolution/
    ├── src/*.py
    ├── notebooks/
    ├── requirements.txt
    └── README.md   (this file)
```

Scripts auto-detect `student_resource/` as three directories up from `src/` (`--repo-root`
overrides). Run CLI commands **from this directory** so `python -m src.<script>` resolves.

## Command-line run order (alternative to the notebook)

```bash
pip install -r requirements.txt

python -m src.normalize --split train --split test
python -m src.blocking --split train --out ../../data_processed/candidate_pairs_train.tsv --report-recall
python -m src.features --split train --candidates ../../data_processed/candidate_pairs_train.tsv
python -m src.train_gbdt --features ../../data_processed/features_train.parquet
python -m src.laya_finetune --stage prepare
torchrun --standalone --nproc_per_node=2 -m src.laya_finetune --stage train --role english
torchrun --standalone --nproc_per_node=2 -m src.laya_finetune --stage train --role multilingual
python -m src.ensemble --features ../../data_processed/features_train.parquet
python -m src.predict --stack-model logistic        # or gbdt_alt / gbdt_only, whichever ensemble.py reports best
```

With a single GPU, use `python -m src.laya_finetune --stage train --role <role>` instead of
`torchrun`.

Validate from `student_resource/`:

```bash
python3 utils/validate_submission.py --matching output/matching_results.tsv \
    --candidate output/candidate_pairs.tsv --test-dir dataset/test
```

## How it scales to the real data (~12.5M train records)

- **Normalization** runs row-parallel across all CPU cores.
- **Blocking** uses TF-IDF vectors (name character 4-grams; address words, which carry pin, city
  and street) and a multithreaded sparse top-n matrix product (`sparse_dot_topn`), per country.
  N-grams shared by more than 10k records are dropped. An earlier Python inverted-index version
  exhausted Kaggle's 30 GB.
- **Training uses a sample** of S1 entities (`--max-s1`, default 300k of ~2.2M). Their candidates
  are still searched against the full S2/S3 pool, so hard negatives are realistic. Every training
  step reads the sample back via `common.sampled_ground_truth`, so splits and metrics all refer to
  the same entities. The test set is always processed in full.
- **Features** are computed in 500k-pair chunks, loading only the records those pairs reference.
- **Laya fine-tuning** uses `--max-finetune-entities` (default 6000) entities: all true matches
  plus up to 3 hard negatives each, in both orderings. DDP across both T4s.
- **Laya scoring** covers a shortlist (each S1's top 3 GBDT candidates with prob >= 0.05), with one
  Router per GPU run in parallel.

## Design notes

- **Output format rules** (one row per S1, matches subset of candidates, no duplicate IDs) are
  enforced structurally: `predict.py` seeds every test S1 id before filling matches, and
  `common.write_id_list_tsv` dedupes. Still confirm with the validator.
- **Country is an open set everywhere**: blocking partitions by whatever `country` values appear;
  only the legal-suffix dictionary branches France vs. everything else.
- **No leakage in stacking**: the GBDT trains on 80% of sampled entities. `ensemble.py` trains
  its stacker only on the other 20% (the GBDT never saw them), halved into stack-train and
  stack-eval. Laya fine-tuning uses only the GBDT's training pool, so `laya_prob` isn't overfit on
  those entities either.
- **GBDT-only fallback**: `ensemble.py` reports the GBDT-only baseline on the same stack-eval
  entities. If the Laya ensemble doesn't beat it, run `predict.py --stack-model gbdt_only`, which
  skips Laya entirely.
- **Laya calibration uses hard 0/1 labels.** The ground truth has no per-pair uncertainty to use
  as soft targets. The RLCD objective and post-hoc temperature fit still work on one-hot targets.
- **Heuristics worth eyeballing on real rows**: DBA splitting and `guess_city` in `normalize.py`.

## What this pipeline does NOT do

- No external data, API, or lookup of any kind (geocoding, business registries, etc.). Nothing
  here calls anything but Hugging Face Hub, to download the two base Laya checkpoints
  (Apache-2.0, 421M / 322M parameters, under the 8B cap).
- No hyperparameter search: LightGBM and RLCD settings are reasonable starting points exposed as
  arguments, not tuned against a score.


In [ ]:
%%writefile $PIPELINE_DIR/src/__init__.py
# (intentionally empty)


In [ ]:
%%writefile $PIPELINE_DIR/src/common.py
"""Shared paths, IO helpers, and scoring utilities used by every script in this pipeline.

NOTE ON EXECUTION: authored on a machine with no GPU. The non-GPU stages (normalize/blocking/
features/train_gbdt) have since been smoke-tested end-to-end against a small synthetic dataset
shaped like the real challenge files, which is how the Unicode combining-mark bug in
`strip_punctuation` below was actually caught -- static reading alone missed it. Laya fine-tuning
and inference (laya_finetune.py's --stage train, ensemble.py/predict.py's Laya scoring) still have
not been run anywhere; those need a real GPU and the actual competition data. Keep spot-checking
outputs on the GPU machine (print a normalized sample, run report_blocking_recall) before trusting
numbers on the real dataset -- a 20-row synthetic smoke test proves the code paths run and do
something sensible, not that every heuristic is well-tuned at full scale.
"""
from __future__ import annotations

import argparse
import random
import re
import unicodedata
from pathlib import Path
from typing import Dict, Iterable, List, Set, Tuple

import pandas as pd

# --------------------------------------------------------------------------------------- paths
#
# src/common.py -> src -> business_entity_resolution -> code -> student_resource
# PROJECT_ROOT is the student_resource/ directory: it holds dataset/, and is where output/,
# data_processed/ and models/ get created. This matches where utils/validate_submission.py
# expects to be run from (its own docstring: "Run this ... from the student_resource/ directory").
SRC_DIR = Path(__file__).resolve().parent
PROJECT_ROOT = SRC_DIR.parents[2]

SPLITS = ("train", "test")
SOURCE_KEYS = ("source1", "source2", "source3")

SOURCE_FILENAMES = {
    "train": {"source1": "train_source1.tsv", "source2": "train_source2.tsv",
              "source3": "train_source3.tsv", "ground_truth": "train_ground_truth.tsv"},
    "test": {"source1": "test_source1.tsv", "source2": "test_source2.tsv",
             "source3": "test_source3.tsv"},
}

def add_repo_root_arg(parser: argparse.ArgumentParser) -> None:
    parser.add_argument(
        "--repo-root", type=Path, default=PROJECT_ROOT,
        help="Directory containing dataset/ (and where output/, data_processed/, models/ are "
             "created). Default: auto-detected student_resource/ directory.",
    )


def require(path, produced_by: str) -> Path:
    """Fail with a readable message when a step's input is missing, naming the earlier step that
    writes it. Without this, a step whose predecessor crashed (e.g. out of memory) fails with a
    bare FileNotFoundError that points at the wrong step."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"{path} does not exist. It is written by {produced_by} -- run that step first and check "
            f"its output for errors (a crashed earlier step is the usual cause).")
    return path


def dataset_dir(repo_root: Path, split: str) -> Path:
    assert split in SPLITS, split
    return Path(repo_root) / "dataset" / split


def data_processed_dir(repo_root: Path) -> Path:
    p = Path(repo_root) / "data_processed"
    p.mkdir(parents=True, exist_ok=True)
    return p


def output_dir(repo_root: Path) -> Path:
    p = Path(repo_root) / "output"
    p.mkdir(parents=True, exist_ok=True)
    return p


def models_dir(repo_root: Path) -> Path:
    p = Path(repo_root) / "models"
    p.mkdir(parents=True, exist_ok=True)
    return p


# --------------------------------------------------------------------------------------- raw IO

_READ_CHUNK_ROWS = 200_000


def _read_tsv_with_progress(path, columns=None, ids=None) -> pd.DataFrame:
    """Every column as `str`, no NaN coercion (an empty field stays ""), read in chunks so large
    files show a live row count instead of a silent multi-minute pause. `columns` limits which
    columns are kept and `ids` which entity_id rows are kept; at ~12.5M train records, loading
    only what a step needs is the difference between fitting in memory and not."""
    from tqdm.auto import tqdm

    path = Path(path)
    chunks = []
    with tqdm(desc=f"read {path.name}", unit="row", unit_scale=True, mininterval=1.0) as bar:
        for chunk in pd.read_csv(path, sep="\t", dtype=str, keep_default_na=False, na_filter=False,
                                 usecols=columns, chunksize=_READ_CHUNK_ROWS):
            bar.update(len(chunk))
            if ids is not None:
                chunk = chunk[chunk["entity_id"].isin(ids)]
            chunks.append(chunk)
    if chunks:
        df = pd.concat(chunks, ignore_index=True)
    else:
        df = pd.read_csv(path, sep="\t", dtype=str, keep_default_na=False, usecols=columns, nrows=0)
    df.columns = [c.strip() for c in df.columns]
    return df


def read_source_tsv(path) -> pd.DataFrame:
    """Read one source/ground-truth TSV. The "null"/"NaN"/"N/A" *strings* some address fields
    contain are left untouched here (they are real content; normalize.py strips them)."""
    return _read_tsv_with_progress(path)


def load_split_sources(repo_root: Path, split: str) -> Dict[str, pd.DataFrame]:
    d = dataset_dir(repo_root, split)
    names = SOURCE_FILENAMES[split]
    out = {key: read_source_tsv(require(d / names[key], "the dataset setup (notebook Section 0.3)"))
           for key in SOURCE_KEYS}
    if split == "train":
        out["ground_truth"] = read_source_tsv(d / names["ground_truth"])
    return out


# --------------------------------------------------------------------------------- processed IO

def normalized_path(repo_root: Path, split: str, key: str) -> Path:
    return data_processed_dir(repo_root) / f"{split}_{key}_normalized.tsv"


def load_normalized(repo_root: Path, split: str, key: str, columns=None, ids=None) -> pd.DataFrame:
    return _read_tsv_with_progress(require(normalized_path(repo_root, split, key),
                                           f"normalize.py (notebook Section 1) for --split {split}"),
                                   columns=columns, ids=ids)


def sampled_ground_truth(repo_root: Path, candidates_path=None) -> pd.DataFrame:
    """Train ground truth restricted to the S1 entities blocking sampled (every sampled S1 has a
    row in candidate_pairs_train.tsv, even with no candidates). Train, Laya, ensemble and the
    recall report all use this, so "val" and every metric refer to the same sampled entities --
    an unsampled entity would otherwise count as an empty prediction and skew F_0.5."""
    candidates_path = candidates_path or (data_processed_dir(repo_root) / "candidate_pairs_train.tsv")
    sampled = set(pd.read_csv(require(candidates_path, "blocking.py --split train (notebook Section 2)"),
                              sep="\t", dtype=str, usecols=["source1_entity_id"], keep_default_na=False)
                  ["source1_entity_id"])
    gt = read_source_tsv(require(dataset_dir(repo_root, "train") / SOURCE_FILENAMES["train"]["ground_truth"],
                                 "the dataset setup (notebook Section 0.3)"))
    return gt[gt["source1_entity_id"].isin(sampled)].reset_index(drop=True)


def write_normalized(df: pd.DataFrame, repo_root: Path, split: str, key: str) -> Path:
    path = normalized_path(repo_root, split, key)
    df.to_csv(path, sep="\t", index=False)
    return path


# ------------------------------------------------------- id-list TSV (the submission-file format)

def write_id_list_tsv(mapping: Dict[str, Iterable[str]], path, id_col: str, list_col: str) -> None:
    """Write the exact (source1_entity_id, comma-joined candidate/match IDs) TSV format that
    utils/validate_submission.py checks: one row per key of `mapping`, no quoting, IDs
    deduplicated, empty string (not "None"/"nan") for an entity with no matches."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8", newline="") as f:
        f.write(f"{id_col}\t{list_col}\n")
        for s1_id, ids in mapping.items():
            id_list = sorted(set(ids))
            f.write(f"{s1_id}\t{','.join(id_list)}\n")


def read_id_list_tsv(path) -> Dict[str, Set[str]]:
    path = Path(path)
    out: Dict[str, Set[str]] = {}
    with open(path, encoding="utf-8") as f:
        next(f, None)  # header
        for line in f:
            line = line.rstrip("\n")
            if not line:
                continue
            s1, _, rest = line.partition("\t")
            out[s1] = set(rest.split(",")) if rest.strip() else set()
    return out




# --------------------------------------------------------------------- stratified S1-level split

def stratified_split_by_s1(ground_truth: pd.DataFrame, val_frac: float = 0.2, seed: int = 42,
                            ) -> Tuple[Set[str], Set[str]]:
    """Split S1 entity IDs (whole entities, not candidate-pair rows) into train/val sets,
    stratified by singleton vs. has-a-match.

    Splitting by entity rather than by row is the point: every (S1, candidate) pair for a given
    S1 entity has to land on the same side of the split, otherwise the model implicitly sees part
    of a validation entity's own pairs during training (leakage), which inflates validation
    F_0.5. Stratifying by singleton/non-singleton keeps both classes represented in both splits
    even though most S1 entities in a challenge like this tend to be non-singletons or vice versa.
    """
    ids = ground_truth["source1_entity_id"].tolist()
    matched = ground_truth["matched_entity_ids"].fillna("")
    is_singleton = matched.str.strip().eq("")
    pos_ids = sorted(i for i, s in zip(ids, is_singleton) if not s)
    neg_ids = sorted(i for i, s in zip(ids, is_singleton) if s)

    rng = random.Random(seed)

    def _split(group: List[str]) -> Tuple[Set[str], Set[str]]:
        group = list(group)
        rng.shuffle(group)
        n_val = int(round(len(group) * val_frac))
        return set(group[n_val:]), set(group[:n_val])

    pos_train, pos_val = _split(pos_ids)
    neg_train, neg_val = _split(neg_ids)
    return pos_train | neg_train, pos_val | neg_val


# ---------------------------------------------------------------------------- F_0.5 (macro) score

def f_beta_per_entity(pred: Dict[str, Set[str]], truth: Dict[str, Set[str]], beta: float = 0.5,
                       ) -> Dict[str, float]:
    """Per-S1-entity F_beta, computed exactly as the challenge scores it: a correctly-predicted
    empty list scores 1.0, an incorrectly non-empty prediction on a true singleton scores 0.0.
    Iterates over `truth`'s keys, so every S1 entity in the ground truth / eval set gets a score
    even when `pred` has no row for it (treated as an empty prediction)."""
    beta2 = beta * beta
    scores: Dict[str, float] = {}
    for s1, true_ids in truth.items():
        pred_ids = pred.get(s1, set())
        if not true_ids and not pred_ids:
            scores[s1] = 1.0
            continue
        if not pred_ids or not true_ids:
            scores[s1] = 0.0
            continue
        tp = len(pred_ids & true_ids)
        if tp == 0:
            scores[s1] = 0.0
            continue
        precision = tp / len(pred_ids)
        recall = tp / len(true_ids)
        denom = beta2 * precision + recall
        scores[s1] = 0.0 if denom == 0 else (1 + beta2) * precision * recall / denom
    return scores


def macro_f_beta(pred: Dict[str, Set[str]], truth: Dict[str, Set[str]], beta: float = 0.5) -> float:
    scores = f_beta_per_entity(pred, truth, beta)
    return sum(scores.values()) / len(scores) if scores else 0.0


def ground_truth_map(ground_truth: pd.DataFrame) -> Dict[str, Set[str]]:
    """{source1_entity_id: set(matched_entity_ids)} from the ground-truth TSV, empty set for a
    singleton. Shared by train_gbdt.py, laya_finetune.py and ensemble.py so positive/negative
    labeling is defined identically everywhere it's used."""
    truth: Dict[str, Set[str]] = {}
    for row in ground_truth.itertuples(index=False):
        row_d = row._asdict()
        raw = row_d["matched_entity_ids"]
        truth[row_d["source1_entity_id"]] = set(raw.split(",")) if raw and raw.strip() else set()
    return truth


def precision_recall_macro(pred: Dict[str, Set[str]], truth: Dict[str, Set[str]]) -> Tuple[float, float]:
    """Micro-averaged precision/recall over all (S1, candidate) pairs, as a companion to the
    macro F_0.5 above (which the challenge actually scores on) — useful for error analysis."""
    tp = fp = fn = 0
    for s1, true_ids in truth.items():
        pred_ids = pred.get(s1, set())
        tp += len(pred_ids & true_ids)
        fp += len(pred_ids - true_ids)
        fn += len(true_ids - pred_ids)
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    return precision, recall


# --------------------------------------------------------------------------------- script detection

_DEVANAGARI_RE = re.compile(r"[ऀ-ॿ]")
_TAMIL_RE = re.compile(r"[஀-௿]")
_LATIN_RE = re.compile(r"[A-Za-z]")


def detect_script(text: str) -> str:
    """Cheap, dependency-free script classifier over Unicode ranges. Returns one of
    'latin', 'devanagari', 'tamil', 'mixed' (no script has >=60% of the letters seen), or
    'other' (no letters from any tracked script — e.g. pure digits/punctuation, or a script
    this challenge doesn't call out such as Chinese/Arabic, which is intentionally bucketed
    as 'other' rather than mis-labelled 'latin')."""
    if not text:
        return "other"
    counts = {
        "devanagari": len(_DEVANAGARI_RE.findall(text)),
        "tamil": len(_TAMIL_RE.findall(text)),
        "latin": len(_LATIN_RE.findall(text)),
    }
    total = sum(counts.values())
    if total == 0:
        return "other"
    dominant, n = max(counts.items(), key=lambda kv: kv[1])
    return dominant if n / total >= 0.6 else "mixed"


# --------------------------------------------------------------------------------- misc text util

_WS_RE = re.compile(r"\s+")


def collapse_ws(text: str) -> str:
    return _WS_RE.sub(" ", text).strip()


# Unicode general-category prefixes to KEEP when stripping punctuation: L* (letters), M* (marks --
# combining vowel signs and the virama/halant that Devanagari, Tamil, and most other Indic/complex
# scripts build words out of), N* (digits).
_KEEP_CATEGORY_PREFIXES = ("L", "M", "N")


def strip_punctuation(text: str, keep_chars: str = "") -> str:
    """Replace every character that is not a letter/mark/digit, whitespace, or in `keep_chars`
    with a single space.

    This is deliberately NOT `re.sub(r"[^\\w\\s]", " ", text)`: Python's `\\w` matches Unicode
    letters and digits but NOT combining marks (category Mn/Mc) -- and Devanagari/Tamil vowel
    signs and the virama/halter are combining marks, not standalone letters. A `\\w`-based strip
    silently deletes them, corrupting every word that uses one (i.e. most Devanagari/Tamil text)
    into fragments -- e.g. "शर्मा" (Sharma) becomes "शर म" (this was caught by actually running
    normalize.py against a synthetic Devanagari-name row, not by reading the regex).
    """
    return "".join(
        ch if ch.isspace() or ch in keep_chars or unicodedata.category(ch).startswith(_KEEP_CATEGORY_PREFIXES)
        else " "
        for ch in text
    )


In [ ]:
%%writefile $PIPELINE_DIR/src/normalize.py
"""Step 1: normalize business_name and business_address across all six source files.

Not executed here (no local run per the project constraints) — read the code back and spot-check
it on the GPU machine before trusting it; see the "NEEDS GPU-MACHINE VERIFICATION" notes below for
the specific spots this file cannot self-check offline (regexes were written against the documented
noise patterns, not against real rows).

Run (from code/business_entity_resolution/):
    python -m src.normalize --split train
    python -m src.normalize --split test
    python -m src.normalize --split train --split test   # both, default

Writes data_processed/{split}_{source1,source2,source3}_normalized.tsv with columns:
    entity_id, business_name, business_address, country,
    name_norm, legal_name_norm, trade_name_norm, has_dba,
    address_norm, pin_code, city_guess,
    name_script, address_script
"""
from __future__ import annotations

import argparse
import multiprocessing
import os
import re
from typing import Dict, Tuple

import pandas as pd
from tqdm.auto import tqdm

from src import common

# --------------------------------------------------------------------------- legal-suffix dicts
#
# Country-aware and *not* merged: France's "SA"/"SAS"/"SARL" must never be expanded by the
# US/India dict (and vice versa) or e.g. a French "SA" could collide with an unrelated US token.
# Every pattern is a whole-word regex, so "sas" cannot match inside "sasu" etc.

_US_INDIA_SUFFIXES: Dict[str, str] = {
    r"\bcorpn\b": "corporation",
    r"\bcorp\b": "corporation",
    r"\bltd\b": "limited",
    r"\bpvt\b": "private",
    r"\bpriv\b": "private",
    r"\binc\b": "incorporated",
    r"\bincorp\b": "incorporated",
    r"\bco\b": "company",
    r"\bllc\b": "limited liability company",
    r"\bllp\b": "limited liability partnership",
    r"\bplc\b": "public limited company",
}

_FRANCE_SUFFIXES: Dict[str, str] = {
    r"\bsasu\b": "societe par actions simplifiee unipersonnelle",
    r"\bsas\b": "societe par actions simplifiee",
    r"\bsarl\b": "societe a responsabilite limitee",
    r"\beurl\b": "entreprise unipersonnelle a responsabilite limitee",
    r"\bsci\b": "societe civile immobiliere",
    r"\bsa\b": "societe anonyme",
}

_SUFFIX_GROUPS = {"us_india": _US_INDIA_SUFFIXES, "france": _FRANCE_SUFFIXES}
_COMPILED_SUFFIX_GROUPS = {
    group: [(re.compile(pat, re.IGNORECASE), repl) for pat, repl in mapping.items()]
    for group, mapping in _SUFFIX_GROUPS.items()
}


def country_suffix_group(country: str) -> str:
    """Bucket an open-set country label into a legal-suffix dictionary. Only 'France' (case
    insensitive) gets the France dict; every other value -- including 'US', 'India', and any
    unseen future country -- gets the US/India dict, which is the only generic one the challenge
    documents. This does NOT filter or branch on country being one of a fixed set anywhere else
    in the pipeline; it only decides which suffix expansion table to apply."""
    c = (country or "").strip().lower()
    return "france" if c == "france" else "us_india"


def expand_legal_suffixes(name_lower: str, country: str) -> str:
    group = country_suffix_group(country)
    for pattern, replacement in _COMPILED_SUFFIX_GROUPS[group]:
        name_lower = pattern.sub(replacement, name_lower)
    return name_lower


# --------------------------------------------------------------------------------- DBA detection
#
# "X dba Y" / "X d/b/a Y" / "X doing business as Y" / "X trading as Y" / "X t/a Y".
# NEEDS GPU-MACHINE VERIFICATION: run this against a sample of real business_name values and
# check for false positives (e.g. a legitimate name containing "as" as a normal word right after
# something that looks like a legal-entity boundary) before trusting it at scale.
_DBA_RE = re.compile(
    r"^(?P<legal>.+?)\s+(?:d/?\.?b/?\.?a\.?|doing\s+business\s+as|trading\s+as|t/a)\s+(?P<trade>.+)$",
    re.IGNORECASE,
)


def split_dba(raw_name: str) -> Tuple[str, str, bool]:
    """Return (legal_part, trade_part, has_dba). When no DBA pattern is found, both parts equal
    the input and has_dba is False."""
    name = common.collapse_ws(raw_name or "")
    m = _DBA_RE.match(name)
    if not m:
        return name, name, False
    return m.group("legal").strip(), m.group("trade").strip(), True


# ------------------------------------------------------------------------------- name normalizer

_AMP_RE = re.compile(r"&")


def normalize_name_core(raw: str, country: str) -> str:
    if not raw:
        return ""
    text = raw.lower()
    text = _AMP_RE.sub(" and ", text)
    text = common.strip_punctuation(text)
    text = common.collapse_ws(text)
    text = expand_legal_suffixes(text, country)
    text = common.collapse_ws(text)
    return text


def normalize_name_fields(raw_name: str, country: str) -> Dict[str, object]:
    legal_raw, trade_raw, has_dba = split_dba(raw_name)
    legal_norm = normalize_name_core(legal_raw, country)
    trade_norm = normalize_name_core(trade_raw, country) if has_dba else legal_norm
    # name_norm is the general-purpose field blocking/features use by default: the legal name,
    # since it's what's present for every record (trade name only differs when has_dba).
    return {
        "name_norm": legal_norm,
        "legal_name_norm": legal_norm,
        "trade_name_norm": trade_norm,
        "has_dba": has_dba,
    }


# ---------------------------------------------------------------------------- address normalizer
#
# Address abbreviations are treated as generic (not country-gated): the challenge describes
# Rd/St/Ave-style abbreviations without tying them to a specific country, unlike the legal
# suffixes, which explicitly must not be conflated across countries.
_ADDRESS_ABBR: Dict[str, str] = {
    r"\brd\b": "road", r"\bst\b": "street", r"\bave\b": "avenue", r"\bblvd\b": "boulevard",
    r"\bln\b": "lane", r"\bdr\b": "drive", r"\bapt\b": "apartment", r"\bfl\b": "floor",
    r"\bste\b": "suite", r"\bhwy\b": "highway", r"\bpl\b": "place", r"\bct\b": "court",
    r"\bsq\b": "square", r"\bter\b": "terrace", r"\bpkwy\b": "parkway", r"\bcir\b": "circle",
    r"\bmkt\b": "market", r"\bnr\b": "near", r"\bopp\b": "opposite", r"\bbldg\b": "building",
}
_COMPILED_ADDRESS_ABBR = [(re.compile(p, re.IGNORECASE), r) for p, r in _ADDRESS_ABBR.items()]

# Literal placeholder tokens that show up as real field *content* (not true NaN, which the TSV
# reader already keeps as "" via keep_default_na=False) — strip them as standalone tokens so
# "123 Main St, null, 10001" doesn't leave a stray "null" city_guess candidate downstream.
_NULL_TOKEN_RE = re.compile(r"(?i)(?<![a-z])(null|nan|n/?a|none)(?![a-z])")

_INDIA_PIN_RE = re.compile(r"\b\d{6}\b")
_GENERIC_POSTAL_RE = re.compile(r"\b\d{5}(?:-\d{4})?\b")


def extract_postal_code(address: str) -> str:
    """India PIN (6 digits) is checked first since it's the more specific pattern; otherwise a
    5-digit (optionally +4) run covers both US ZIP and French postal codes -- disambiguating
    which of those it is isn't needed here, only capturing the digits for blocking/matching.
    Takes the LAST match in the string, since postal codes conventionally trail an address."""
    matches = _INDIA_PIN_RE.findall(address)
    if matches:
        return matches[-1]
    matches = _GENERIC_POSTAL_RE.findall(address)
    return matches[-1] if matches else ""


_DIGITS_RE = re.compile(r"\d+")


def guess_city(address_norm: str, pin_code: str) -> str:
    """Best-effort heuristic, NOT a gazetteer lookup (none is available/allowed for this
    challenge). Two address shapes are common and disagree about where the city sits relative to
    the PIN code's own comma segment:
      - "<street>, <city>, <state> <pin>" or "<street>, <city>, <pin>" (US/India-style): the city
        is the segment BEFORE the one holding the pin.
      - "<street>, <pin> <city>" (France-style, e.g. "12 Rue de Paris, 75001 Paris"): the city is
        IN THE SAME segment as the pin, and the segment before it is the street.
    Disambiguate with one signal: does the segment before the pin's segment look like a street
    (starts with a number, e.g. a house/building number)? If so, prefer extracting the city out of
    the pin's own segment; otherwise treat that previous segment as the city, as before. Falls
    back to the second-to-last comma segment when no PIN was found. NEEDS GPU-MACHINE
    VERIFICATION on the real data: landmark-style addresses ("near SBI ATM") and municipal-
    numbering addresses (no commas at all) will likely still return "" or an imperfect token,
    which is acceptable -- this is a secondary blocking signal, not the only one -- but confirm
    the actual hit rate on real rows, not this docstring.
    """
    segments = [s.strip() for s in address_norm.split(",") if s.strip()]
    if not segments:
        return ""
    if pin_code:
        for i, seg in enumerate(segments):
            if pin_code in seg:
                same_segment_city = _DIGITS_RE.sub("", seg).strip()
                prev_looks_like_street = i > 0 and bool(re.match(r"^\d", segments[i - 1]))
                if same_segment_city and (prev_looks_like_street or i == 0):
                    return same_segment_city
                return segments[i - 1] if i > 0 else ""
    return segments[-2] if len(segments) >= 2 else ""


def normalize_address_fields(raw_address: str) -> Dict[str, str]:
    text = raw_address or ""
    text = _NULL_TOKEN_RE.sub(" ", text)
    text = text.lower()
    pin_code = extract_postal_code(text)
    for pattern, replacement in _COMPILED_ADDRESS_ABBR:
        text = pattern.sub(replacement, text)
    # keep commas (city/state segmentation depends on them); drop other punctuation
    text = common.strip_punctuation(text, keep_chars=",")
    text = re.sub(r"\s*,\s*", ", ", text)
    text = common.collapse_ws(text)
    city_guess = guess_city(text, pin_code)
    return {"address_norm": text, "pin_code": pin_code, "city_guess": city_guess}


# --------------------------------------------------------------------------------------- driver

_OUTPUT_COLS = [
    "entity_id", "business_name", "business_address", "country",
    "name_norm", "legal_name_norm", "trade_name_norm", "has_dba",
    "address_norm", "pin_code", "city_guess", "name_script", "address_script",
]


def normalize_row(row) -> tuple:
    entity_id, raw_name, raw_address, country = row
    raw_name = raw_name or ""
    raw_address = raw_address or ""
    country = (country or "").strip()
    fields = {
        "entity_id": entity_id,
        "business_name": raw_name,
        "business_address": raw_address,
        "country": country,
        **normalize_name_fields(raw_name, country),
        **normalize_address_fields(raw_address),
        "name_script": common.detect_script(raw_name),
        "address_script": common.detect_script(raw_address),
    }
    return tuple(fields[c] for c in _OUTPUT_COLS)


def normalize_frame(df: pd.DataFrame, label: str = "", workers: int = None) -> pd.DataFrame:
    """Row-parallel across `workers` processes (default: every CPU core). Pure-Python regex work
    like this is CPU-bound and doesn't benefit from a GPU; using all cores is the real speedup."""
    workers = workers or os.cpu_count() or 1
    rows = list(zip(df["entity_id"], df["business_name"], df["business_address"], df["country"]))
    bar = tqdm(total=len(rows), desc=f"normalize {label}", unit="row", unit_scale=True, mininterval=1.0)
    if workers == 1 or len(rows) < 20_000:
        out = []
        for row in rows:
            out.append(normalize_row(row))
            bar.update(1)
    else:
        out = []
        with multiprocessing.get_context("fork").Pool(workers) as pool:
            for result in pool.imap(normalize_row, rows, chunksize=2_000):
                out.append(result)
                bar.update(1)
    bar.close()
    return pd.DataFrame(out, columns=_OUTPUT_COLS)


def run(repo_root, splits, skip_existing: bool = False):
    """skip_existing: reuse normalized files already on disk (e.g. from a previous Kaggle session)
    instead of recomputing them. Only safe if they were produced by the current normalize.py."""
    for split in splits:
        for key in common.SOURCE_KEYS:
            if skip_existing and common.normalized_path(repo_root, split, key).exists():
                print(f"[normalize] {split}/{key}: reusing existing {common.normalized_path(repo_root, split, key)}")
                continue
            # one source at a time: holding all three raw splits at once needlessly doubles peak memory
            raw = common.read_source_tsv(common.require(
                common.dataset_dir(repo_root, split) / common.SOURCE_FILENAMES[split][key],
                "the dataset setup (notebook Section 0.3)"))
            print(f"[normalize] {split}/{key}: {len(raw)} rows")
            normalized = normalize_frame(raw, label=f"{split}/{key}")
            del raw
            out_path = common.write_normalized(normalized, repo_root, split, key)
            print(f"[normalize]   -> {out_path}")


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    common.add_repo_root_arg(parser)
    parser.add_argument("--split", action="append", choices=list(common.SPLITS),
                         help="Repeatable. Default: both train and test.")
    args = parser.parse_args()
    splits = args.split or list(common.SPLITS)
    run(args.repo_root, splits)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile $PIPELINE_DIR/src/blocking.py
"""Step 2: blocking / candidate generation, built to scale to ~10M+ records.

For each S1 entity, finds its most similar S2/S3 records within the same `country` (an OPEN SET:
partitions are whatever country values appear in the data, never a hardcoded list), using two
TF-IDF views searched with a multithreaded sparse top-n matrix product (sparse_dot_topn):
  - name:    character 4-grams of name_norm (typo/abbreviation tolerant)
  - address: word tokens of address_norm (pin/postal codes, city, street -- this is the
             pin/city blocking signal)
Per S1 entity the two views' candidates are unioned, scored by name_sim + address_sim, and the
top K kept.

Why not Python inverted indices: the train split alone is ~12.5M records. Python dict/set/
Counter structures per record exhausted Kaggle's 30 GB and would take many hours to loop over
2.2M S1 entities. Sparse CSR matrices hold the same information in a few GB, and the top-n product
runs in C++ across every core. N-grams shared by more than MAX_DF records are dropped: they
carry no signal and would dominate both memory and compute.

TRAIN split: blocking (and everything downstream) uses a random sample of --max-s1 S1 entities
(default 300k of ~2.2M). Their candidates are still searched against the FULL S2/S3 pool, so hard
negatives stay as hard as at test time. The rest of the pipeline reads the sample back from
candidate_pairs_train.tsv (common.sampled_ground_truth). TEST split: always every S1 entity.

Run (from code/business_entity_resolution/):
    python -m src.blocking --split train --out ../../data_processed/candidate_pairs_train.tsv --report-recall
    python -m src.blocking --split test      # predict.py does this itself too
"""
from __future__ import annotations

import argparse
import os
from typing import Dict, List, Optional, Set

import numpy as np
import pandas as pd
from scipy import sparse
from tqdm.auto import tqdm

from src import common

DEFAULT_K = 30
DEFAULT_MAX_TRAIN_S1 = 300_000
NAME_TOP_N = 25
ADDRESS_TOP_N = 15
NAME_THRESHOLD = 0.1
ADDRESS_THRESHOLD = 0.1
MAX_DF = 10_000          # drop n-grams/tokens appearing in more records than this (per country)
S1_CHUNK = 20_000
BLOCKING_COLUMNS = ["entity_id", "country", "name_norm", "address_norm"]


def _vectorizer(view: str):
    from sklearn.feature_extraction.text import TfidfVectorizer

    if view == "name":
        return TfidfVectorizer(analyzer="char_wb", ngram_range=(4, 4), max_df=MAX_DF,
                               sublinear_tf=True, dtype=np.float32)
    return TfidfVectorizer(analyzer="word", token_pattern=r"(?u)\b\w+\b", max_df=MAX_DF,
                           sublinear_tf=True, dtype=np.float32)


def _topn_view(view: str, pool_texts: List[str], s1_texts: List[str], top_n: int,
               threshold: float, label: str) -> Optional[sparse.csr_matrix]:
    """(n_s1 x n_pool) sparse matrix of each S1 row's top_n cosine similarities against the pool,
    or None when the view has no usable vocabulary for this partition (e.g. every address empty)."""
    from sparse_dot_topn import sp_matmul_topn

    vec = _vectorizer(view)
    try:
        pool_m = vec.fit_transform(pool_texts)
    except ValueError:  # empty vocabulary / everything pruned by max_df
        return None
    pool_t = pool_m.T.tocsr()
    del pool_m
    s1_m = vec.transform(s1_texts).tocsr()
    n_threads = os.cpu_count() or 1
    parts = []
    for start in tqdm(range(0, s1_m.shape[0], S1_CHUNK), desc=f"{label} {view} top-{top_n}",
                      unit="chunk", leave=False):
        parts.append(sp_matmul_topn(s1_m[start:start + S1_CHUNK], pool_t, top_n=top_n,
                                    threshold=threshold, n_threads=n_threads).tocsr())
    return sparse.vstack(parts, format="csr") if parts else None


def _blocking_partition(s1_ids, s1_names, s1_addrs, pool_ids, pool_names, pool_addrs, k: int,
                        label: str) -> Dict[str, List[str]]:
    views = [
        _topn_view("name", pool_names, s1_names, NAME_TOP_N, NAME_THRESHOLD, label),
        _topn_view("address", pool_addrs, s1_addrs, ADDRESS_TOP_N, ADDRESS_THRESHOLD, label),
    ]
    views = [v for v in views if v is not None]
    out: Dict[str, List[str]] = {}
    if not views:
        return {s1: [] for s1 in s1_ids}
    combined = views[0] if len(views) == 1 else (views[0] + views[1]).tocsr()
    indptr, indices, data = combined.indptr, combined.indices, combined.data
    for row, s1 in enumerate(s1_ids):
        lo, hi = indptr[row], indptr[row + 1]
        if hi - lo > k:
            keep = np.argpartition(-data[lo:hi], k)[:k]
            order = keep[np.argsort(-data[lo:hi][keep])]
        else:
            order = np.argsort(-data[lo:hi])
        out[s1] = [pool_ids[j] for j in indices[lo:hi][order]]
    return out


def generate_candidates(s1_df: pd.DataFrame, s2_df: pd.DataFrame, s3_df: pd.DataFrame,
                        k: int = DEFAULT_K) -> Dict[str, List[str]]:
    """{s1_entity_id: [candidate_id, ...]} ranked by name_sim + address_sim, capped at k. Every S1
    row gets a key, even with no candidates (e.g. a country with no S2/S3 records)."""
    pool = pd.concat([s2_df[BLOCKING_COLUMNS], s3_df[BLOCKING_COLUMNS]], ignore_index=True)
    pool_by_country = {c: g for c, g in pool.groupby("country", sort=False)}
    results: Dict[str, List[str]] = {}
    for country, s1_group in tqdm(list(s1_df.groupby("country", sort=False)), desc="blocking countries",
                                  unit="country"):
        s1_ids = s1_group["entity_id"].tolist()
        pool_group = pool_by_country.get(country)
        if pool_group is None or pool_group.empty:
            results.update({s1: [] for s1 in s1_ids})
            continue
        print(f"[blocking] {country}: {len(s1_ids)} S1 vs {len(pool_group)} S2/S3 records")
        results.update(_blocking_partition(
            s1_ids, s1_group["name_norm"].tolist(), s1_group["address_norm"].tolist(),
            pool_group["entity_id"].to_numpy(), pool_group["name_norm"].tolist(),
            pool_group["address_norm"].tolist(), k, label=str(country)))
    return results


def sample_s1(s1_df: pd.DataFrame, max_s1: Optional[int], seed: int) -> pd.DataFrame:
    if max_s1 is None or len(s1_df) <= max_s1:
        return s1_df
    return s1_df.sample(n=max_s1, random_state=seed).reset_index(drop=True)


# ---------------------------------------------------------------------------- recall diagnostics

def report_blocking_recall(candidate_ids: Dict[str, List[str]], ground_truth: pd.DataFrame,
                           eval_ids: Set[str] = None) -> Dict[str, float]:
    """Fraction of true matches that survive into the candidate set -- the recall CEILING for
    everything downstream. `eval_ids` restricts the report to e.g. the held-out val split."""
    truth = common.ground_truth_map(ground_truth)
    total_true = total_recalled = 0
    per_entity_recall = []
    singleton_correct = singleton_total = 0
    for s1, true_ids in truth.items():
        if eval_ids is not None and s1 not in eval_ids:
            continue
        cand_ids = set(candidate_ids.get(s1, ()))
        if not true_ids:
            singleton_total += 1
            singleton_correct += not cand_ids
            continue
        recalled = len(true_ids & cand_ids)
        total_true += len(true_ids)
        total_recalled += recalled
        per_entity_recall.append(recalled / len(true_ids))

    return {
        "n_nonsingleton_entities": len(per_entity_recall),
        "overall_pair_recall": (total_recalled / total_true) if total_true else float("nan"),
        "macro_entity_recall": (sum(per_entity_recall) / len(per_entity_recall)) if per_entity_recall else float("nan"),
        "n_singleton_entities": singleton_total,
        "singleton_empty_candidate_rate": (singleton_correct / singleton_total) if singleton_total else float("nan"),
    }


# --------------------------------------------------------------------------------------- driver

def load_blocking_inputs(repo_root, split: str):
    return [common.load_normalized(repo_root, split, key, columns=BLOCKING_COLUMNS)
            for key in common.SOURCE_KEYS]


def run(repo_root, split: str, k: int, out_path, report_recall: bool, val_frac: float, seed: int,
        max_s1: Optional[int]):
    s1_df, s2_df, s3_df = load_blocking_inputs(repo_root, split)
    if split == "train":
        s1_df = sample_s1(s1_df, max_s1, seed)
    print(f"[blocking] {split}: {len(s1_df)} S1 (after sampling) / {len(s2_df)} S2 / {len(s3_df)} S3, k={k}")
    id_map = generate_candidates(s1_df, s2_df, s3_df, k=k)
    del s2_df, s3_df

    out_path = out_path or (common.output_dir(repo_root) / "candidate_pairs.tsv")
    common.write_id_list_tsv(id_map, out_path, "source1_entity_id", "candidate_entity_ids")
    print(f"[blocking] wrote {out_path} ({len(id_map)} S1 rows, {sum(len(v) for v in id_map.values())} pairs)")

    if report_recall:
        if split != "train":
            raise SystemExit("--report-recall needs ground truth, which only exists for --split train")
        ground_truth = common.sampled_ground_truth(repo_root, out_path)
        _, val_ids = common.stratified_split_by_s1(ground_truth, val_frac=val_frac, seed=seed)
        stats = report_blocking_recall(id_map, ground_truth, eval_ids=val_ids)
        print(f"[blocking] recall on held-out val split ({len(val_ids)} sampled S1 entities):")
        for key, value in stats.items():
            print(f"    {key}: {value}")


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    common.add_repo_root_arg(parser)
    parser.add_argument("--split", choices=list(common.SPLITS), required=True)
    parser.add_argument("--k", type=int, default=DEFAULT_K)
    parser.add_argument("--max-s1", type=int, default=DEFAULT_MAX_TRAIN_S1,
                        help="[train only] S1 entities sampled for training. 0 = all.")
    parser.add_argument("--out", type=str, default=None,
                        help="Output path. Default: output/candidate_pairs.tsv")
    parser.add_argument("--report-recall", action="store_true")
    parser.add_argument("--val-frac", type=float, default=0.2)
    parser.add_argument("--seed", type=int, default=42)
    args = parser.parse_args()
    run(args.repo_root, args.split, args.k, args.out, args.report_recall, args.val_frac, args.seed,
        args.max_s1 or None)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile $PIPELINE_DIR/src/features.py
"""Step 3: pairwise feature engineering for every (S1, candidate) pair in candidate_pairs.tsv.

Vectorized and chunked so it scales to millions of pairs: entity records live in one indexed
DataFrame (not a dict per record), string metrics use rapidfuzz.process.cpdist (C-level,
multi-threaded), and TF-IDF vectorizers are fit ONCE on the unique names involved, then pairs are
scored by row-wise dot products on index arrays.

Run (from code/business_entity_resolution/):
    python -m src.features --split train --candidates ../../data_processed/candidate_pairs_train.tsv

Writes data_processed/features_{split}.parquet, one row per (source1_entity_id,
candidate_entity_id).
"""
from __future__ import annotations

import argparse
from typing import Dict, List

import numpy as np
import pandas as pd
from rapidfuzz import process
from rapidfuzz.distance import JaroWinkler, Levenshtein
from tqdm.auto import tqdm

from src import common

WORD_NGRAM_RANGE = (1, 2)
CHAR_NGRAM_RANGE = (3, 4)
CHUNK_SIZE = 500_000

ENTITY_COLS = [
    "entity_id", "business_name", "business_address", "country",
    "name_norm", "legal_name_norm", "trade_name_norm", "has_dba",
    "address_norm", "pin_code", "city_guess", "name_script",
]

FEATURE_COLUMNS = [
    "source1_entity_id", "candidate_entity_id",
    "name_levenshtein_sim", "name_jaro_winkler", "name_token_jaccard",
    "name_tfidf_cosine_word", "name_tfidf_cosine_char",
    "address_levenshtein_sim", "address_jaro_winkler", "address_token_jaccard",
    "pin_exact_match", "city_exact_match", "country_match", "name_len_ratio",
    "is_source2", "is_source3", "s1_has_dba", "cand_has_dba",
    "dba_best_jaccard", "name_script_match",
]


# --------------------------------------------------------------------------------- entity lookup

def load_entity_table(repo_root, split: str, ids=None) -> pd.DataFrame:
    """Normalized S1/S2/S3 records in one frame indexed by entity_id (IDs are globally unique by
    prefix). Pass `ids` to keep only the records a step references -- the full train split is
    ~12.5M records, far more than any single step needs in memory."""
    ids = set(ids) if ids is not None else None
    frames = [common.load_normalized(repo_root, split, key, columns=ENTITY_COLS, ids=ids)
              for key in common.SOURCE_KEYS]
    return pd.concat(frames, ignore_index=True).set_index("entity_id")


def pair_entity_ids(pairs_df: pd.DataFrame) -> set:
    return set(pairs_df["source1_entity_id"]).union(pairs_df["candidate_entity_id"])


def load_entity_lookup(repo_root, split: str, ids) -> Dict[str, dict]:
    """entity_id -> record dict for just `ids` (the Laya fine-tuning examples' records)."""
    return load_entity_table(repo_root, split, ids=ids).to_dict("index")


def build_pairs_frame(candidate_map) -> pd.DataFrame:
    s1_ids, cand_ids = [], []
    for s1, cands in candidate_map.items():
        for c in cands:
            s1_ids.append(s1)
            cand_ids.append(c)
    return pd.DataFrame({"source1_entity_id": s1_ids, "candidate_entity_id": cand_ids})


# ----------------------------------------------------------------------------- string similarity

def token_jaccard(a: str, b: str) -> float:
    ta, tb = set(a.split()), set(b.split())
    if not ta or not tb:
        return 0.0
    inter = len(ta & tb)
    return inter / len(ta | tb) if inter else 0.0


def _jaccard_many(xs: List[str], ys: List[str]) -> np.ndarray:
    return np.fromiter((token_jaccard(a, b) for a, b in zip(xs, ys)), dtype=np.float32, count=len(xs))


def _pairwise(xs: List[str], ys: List[str], scorer) -> np.ndarray:
    return process.cpdist(xs, ys, scorer=scorer, workers=-1, dtype=np.float32)


def _fit_tfidf(texts: List[str], analyzer: str, ngram_range):
    from sklearn.feature_extraction.text import TfidfVectorizer

    non_empty = [t for t in texts if t]
    if not non_empty:
        return None
    vectorizer = TfidfVectorizer(analyzer=analyzer, ngram_range=ngram_range, min_df=1, dtype=np.float32)
    vectorizer.fit(non_empty)
    return vectorizer.transform(texts).tocsr()


def _rowdot(matrix, ia: np.ndarray, ib: np.ndarray) -> np.ndarray:
    """Cosine between rows ia[i] and ib[i] of an L2-normalized TF-IDF matrix."""
    if matrix is None:
        return np.zeros(len(ia), dtype=np.float32)
    return np.asarray(matrix[ia].multiply(matrix[ib]).sum(axis=1), dtype=np.float32).ravel()


# --------------------------------------------------------------------------------------- driver

def compute_features(pairs_df: pd.DataFrame, table: pd.DataFrame, chunk_size: int = CHUNK_SIZE) -> pd.DataFrame:
    known = pairs_df["source1_entity_id"].isin(table.index) & pairs_df["candidate_entity_id"].isin(table.index)
    missing = int((~known).sum())
    if missing:
        print(f"[features] WARNING: {missing} candidate pairs referenced an entity_id not found in the "
              f"normalized source files -- skipped. Check that candidate_pairs.tsv matches --split.")
    pairs = pairs_df[known].reset_index(drop=True)
    if pairs.empty:
        return pd.DataFrame(columns=FEATURE_COLUMNS)

    involved = pd.unique(np.concatenate([pairs["source1_entity_id"].to_numpy(),
                                         pairs["candidate_entity_id"].to_numpy()]))
    row_of = pd.Series(np.arange(len(involved)), index=involved)
    names = table.loc[involved, "name_norm"].tolist()
    print(f"[features] fitting TF-IDF on {len(names)} unique names")
    word_m = _fit_tfidf(names, "word", WORD_NGRAM_RANGE)
    char_m = _fit_tfidf(names, "char_wb", CHAR_NGRAM_RANGE)

    out = []
    for start in tqdm(range(0, len(pairs), chunk_size), desc="features", unit="chunk"):
        chunk = pairs.iloc[start:start + chunk_size]
        s1_ids = chunk["source1_entity_id"].to_numpy()
        cand_ids = chunk["candidate_entity_id"].to_numpy()
        a = table.loc[s1_ids]
        b = table.loc[cand_ids]
        ia = row_of.loc[s1_ids].to_numpy()
        ib = row_of.loc[cand_ids].to_numpy()

        na, nb = a["name_norm"].tolist(), b["name_norm"].tolist()
        ada, adb = a["address_norm"].tolist(), b["address_norm"].tolist()
        la, lb = a["name_norm"].str.len().to_numpy(), b["name_norm"].str.len().to_numpy()
        longest = np.maximum(la, lb)
        pa, pb = a["pin_code"].to_numpy(), b["pin_code"].to_numpy()
        ca, cb = a["city_guess"].to_numpy(), b["city_guess"].to_numpy()

        dba_best = np.maximum.reduce([
            _jaccard_many(a["legal_name_norm"].tolist(), b["legal_name_norm"].tolist()),
            _jaccard_many(a["legal_name_norm"].tolist(), b["trade_name_norm"].tolist()),
            _jaccard_many(a["trade_name_norm"].tolist(), b["legal_name_norm"].tolist()),
            _jaccard_many(a["trade_name_norm"].tolist(), b["trade_name_norm"].tolist()),
        ])

        out.append(pd.DataFrame({
            "source1_entity_id": s1_ids,
            "candidate_entity_id": cand_ids,
            "name_levenshtein_sim": _pairwise(na, nb, Levenshtein.normalized_similarity),
            "name_jaro_winkler": _pairwise(na, nb, JaroWinkler.normalized_similarity),
            "name_token_jaccard": _jaccard_many(na, nb),
            "name_tfidf_cosine_word": _rowdot(word_m, ia, ib),
            "name_tfidf_cosine_char": _rowdot(char_m, ia, ib),
            "address_levenshtein_sim": _pairwise(ada, adb, Levenshtein.normalized_similarity),
            "address_jaro_winkler": _pairwise(ada, adb, JaroWinkler.normalized_similarity),
            "address_token_jaccard": _jaccard_many(ada, adb),
            "pin_exact_match": (pa != "") & (pa == pb),
            "city_exact_match": (ca != "") & (ca == cb),
            "country_match": (a["country"].str.strip().str.lower().to_numpy()
                              == b["country"].str.strip().str.lower().to_numpy()),
            "name_len_ratio": np.where(longest > 0, np.minimum(la, lb) / np.maximum(longest, 1), 0.0).astype(np.float32),
            "is_source2": chunk["candidate_entity_id"].str.startswith("S2-").to_numpy(),
            "is_source3": chunk["candidate_entity_id"].str.startswith("S3-").to_numpy(),
            "s1_has_dba": a["has_dba"].to_numpy() == "True",
            "cand_has_dba": b["has_dba"].to_numpy() == "True",
            "dba_best_jaccard": dba_best,
            "name_script_match": a["name_script"].to_numpy() == b["name_script"].to_numpy(),
        }))
    return pd.concat(out, ignore_index=True)[FEATURE_COLUMNS]


def run(repo_root, split: str, candidates_path, out_path):
    candidates_path = candidates_path or (common.output_dir(repo_root) / "candidate_pairs.tsv")
    candidate_map = common.read_id_list_tsv(common.require(candidates_path, "blocking.py (notebook Section 2)"))
    pairs_df = build_pairs_frame(candidate_map)
    print(f"[features] {split}: {len(candidate_map)} S1 entities, {len(pairs_df)} candidate pairs")

    table = load_entity_table(repo_root, split, ids=pair_entity_ids(pairs_df))
    features_df = compute_features(pairs_df, table)

    out_path = out_path or (common.data_processed_dir(repo_root) / f"features_{split}.parquet")
    features_df.to_parquet(out_path, index=False)
    print(f"[features] wrote {out_path} ({len(features_df)} rows, {features_df.shape[1]} columns)")


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    common.add_repo_root_arg(parser)
    parser.add_argument("--split", choices=list(common.SPLITS), required=True)
    parser.add_argument("--candidates", type=str, default=None,
                        help="Path to a candidate_pairs.tsv. Default: output/candidate_pairs.tsv")
    parser.add_argument("--out", type=str, default=None,
                        help="Output parquet path. Default: data_processed/features_{split}.parquet")
    args = parser.parse_args()
    run(args.repo_root, args.split, args.candidates, args.out)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile $PIPELINE_DIR/src/train_gbdt.py
"""Step 4: GBDT (LightGBM) binary matcher over the feature table from features.py.

Labels: positive = candidate is in the S1 entity's matched_entity_ids (ground truth); negative =
candidate survived blocking but is NOT a true match (a "hard negative" -- it looks plausible
enough to have passed blocking, unlike a random pair, which is what makes it a useful negative).

Split: stratified by whole S1 entity (src/common.py:stratified_split_by_s1), not by row, so no
candidate pair from a validation entity is seen during training.

Not executed here. This script is written to run as-is on the GPU machine -- it does not need a
GPU (LightGBM here runs on CPU by default), but per the project constraints nothing is run in this
authoring environment. Read it back / spot-check the printed metrics format on the GPU machine.

Run (from code/business_entity_resolution/):
    python -m src.train_gbdt --features data_processed/features_train.parquet

Writes:
    models/gbdt_model.txt                      (LightGBM Booster, text format)
    models/gbdt_feature_columns.json            (the exact feature column order/list used)
    data_processed/gbdt_val_report.json         (precision/recall/macro F_0.5 + threshold sweep)
"""
from __future__ import annotations

import argparse
import json
from typing import Dict, List, Set

import numpy as np
import pandas as pd

from src import common

ID_COLS = ("source1_entity_id", "candidate_entity_id")
DEFAULT_THRESHOLD = 0.5


def feature_columns(features_df: pd.DataFrame) -> List[str]:
    return [c for c in features_df.columns if c not in ID_COLS]


def _bool_and_str_to_numeric(df: pd.DataFrame, cols: List[str]) -> pd.DataFrame:
    """LightGBM wants numeric input; the feature table has a few boolean columns (pin_exact_match,
    is_source2, dba flags, ...) that pandas/pyarrow round-trip as Python bool -- cast everything
    to float32 uniformly rather than special-casing each column name."""
    out = df.copy()
    for c in cols:
        out[c] = out[c].astype("float32")
    return out


def label_pairs(features_df: pd.DataFrame, ground_truth: pd.DataFrame) -> pd.Series:
    truth = common.ground_truth_map(ground_truth)
    positive_keys = {f"{s1}|{c}" for s1, ids in truth.items() for c in ids}
    keys = features_df["source1_entity_id"] + "|" + features_df["candidate_entity_id"]
    return keys.isin(positive_keys)


def build_dataset(repo_root, features_path):
    features_path = features_path or (common.data_processed_dir(repo_root) / "features_train.parquet")
    features_df = pd.read_parquet(common.require(features_path, "features.py (notebook Section 3)"))
    ground_truth = common.sampled_ground_truth(repo_root)
    labels = label_pairs(features_df, ground_truth)
    return features_df, labels, ground_truth


def score_with_gbdt(model, features_df: pd.DataFrame, cols: List[str] = None) -> np.ndarray:
    """Reusable by ensemble.py / predict.py: apply a trained Booster to any features_df with (at
    least) the same feature columns it was trained on."""
    cols = cols or feature_columns(features_df)
    X = _bool_and_str_to_numeric(features_df, cols)[cols]
    return model.predict(X)


def matches_from_scores(features_df: pd.DataFrame, scores: np.ndarray, threshold: float,
                         candidate_ids: Dict[str, Set[str]] = None) -> Dict[str, Set[str]]:
    """Aggregate per-pair scores into per-S1 predicted match sets at a probability threshold.
    `candidate_ids`, when given, is the full candidate map (so S1 entities with candidates but
    zero rows above threshold still get an explicit empty entry instead of being absent)."""
    out: Dict[str, Set[str]] = {s1: set() for s1 in candidate_ids} if candidate_ids else {}
    for s1, cand, score in zip(features_df["source1_entity_id"], features_df["candidate_entity_id"], scores):
        if score >= threshold:
            out.setdefault(s1, set()).add(cand)
    return out


def evaluate(features_df: pd.DataFrame, scores: np.ndarray, ground_truth: pd.DataFrame,
             val_ids: Set[str], thresholds=None) -> dict:
    """Precision/recall/macro-F0.5 at DEFAULT_THRESHOLD, plus a small threshold sweep (for
    diagnostics only -- final threshold selection is ensemble.py's job, on the ensemble's own
    output, not the bare GBDT's)."""
    thresholds = thresholds if thresholds is not None else [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
    val_mask = features_df["source1_entity_id"].isin(val_ids)
    val_df = features_df[val_mask].reset_index(drop=True)
    val_scores = np.asarray(scores)[val_mask.to_numpy()]

    truth = common.ground_truth_map(ground_truth)
    val_truth = {s1: ids for s1, ids in truth.items() if s1 in val_ids}

    sweep = {}
    for t in thresholds:
        pred = matches_from_scores(val_df, val_scores, t, candidate_ids={s1: set() for s1 in val_ids})
        precision, recall = common.precision_recall_macro(pred, val_truth)
        macro_f05 = common.macro_f_beta(pred, val_truth, beta=0.5)
        sweep[str(t)] = {"precision": precision, "recall": recall, "macro_f0_5": macro_f05}

    return {"n_val_entities": len(val_ids), "n_val_pairs": int(val_mask.sum()), "threshold_sweep": sweep}


def train(repo_root, features_path, val_frac: float, seed: int, num_boost_round: int,
          early_stopping_rounds: int, extra_params: dict = None):
    import lightgbm as lgb

    features_df, labels, ground_truth = build_dataset(repo_root, features_path)
    cols = feature_columns(features_df)
    X_all = _bool_and_str_to_numeric(features_df, cols)[cols]
    y_all = labels.astype(int)

    train_ids, val_ids = common.stratified_split_by_s1(ground_truth, val_frac=val_frac, seed=seed)
    train_mask = features_df["source1_entity_id"].isin(train_ids).to_numpy()
    val_mask = features_df["source1_entity_id"].isin(val_ids).to_numpy()

    print(f"[train_gbdt] {train_mask.sum()} train pairs / {val_mask.sum()} val pairs "
          f"({y_all[train_mask].sum()} / {y_all[val_mask].sum()} positive)")

    train_set = lgb.Dataset(X_all[train_mask], label=y_all[train_mask], feature_name=cols)
    val_set = lgb.Dataset(X_all[val_mask], label=y_all[val_mask], feature_name=cols, reference=train_set)

    params = {
        "objective": "binary",
        "metric": ["binary_logloss", "auc"],
        "learning_rate": 0.05,
        "num_leaves": 31,
        "min_data_in_leaf": 20,
        "feature_fraction": 0.9,
        "bagging_fraction": 0.8,
        "bagging_freq": 1,
        "is_unbalance": True,  # true matches are a small minority of blocking candidates
        "verbose": -1,
        "seed": seed,
    }
    if extra_params:
        params.update(extra_params)

    from tqdm.auto import tqdm

    bar = tqdm(total=num_boost_round, desc="lightgbm", unit="iter")

    def _progress(env):
        bar.update(1)

    model = lgb.train(
        params, train_set, num_boost_round=num_boost_round,
        valid_sets=[val_set], valid_names=["val"],
        callbacks=[lgb.early_stopping(early_stopping_rounds), lgb.log_evaluation(100), _progress],
    )
    bar.close()

    val_scores_full = np.zeros(len(features_df), dtype="float32")
    val_scores_full[val_mask] = model.predict(X_all[val_mask], num_iteration=model.best_iteration)
    report = evaluate(features_df, val_scores_full, ground_truth, val_ids)

    models_path = common.models_dir(repo_root)
    model.save_model(str(models_path / "gbdt_model.txt"), num_iteration=model.best_iteration)
    with open(models_path / "gbdt_feature_columns.json", "w") as f:
        json.dump(cols, f, indent=2)
    report_path = common.data_processed_dir(repo_root) / "gbdt_val_report.json"
    with open(report_path, "w") as f:
        json.dump(report, f, indent=2)

    print(f"[train_gbdt] best_iteration={model.best_iteration}")
    print(f"[train_gbdt] wrote {models_path / 'gbdt_model.txt'}")
    print(f"[train_gbdt] val report -> {report_path}")
    print(json.dumps(report, indent=2))
    return model, report


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    common.add_repo_root_arg(parser)
    parser.add_argument("--features", type=str, default=None,
                         help="Path to features_train.parquet. Default: data_processed/features_train.parquet")
    parser.add_argument("--val-frac", type=float, default=0.2)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--num-boost-round", type=int, default=2000)
    parser.add_argument("--early-stopping-rounds", type=int, default=50)
    args = parser.parse_args()
    train(args.repo_root, args.features, args.val_frac, args.seed,
          args.num_boost_round, args.early_stopping_rounds)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile $PIPELINE_DIR/src/laya_finetune.py
"""Step 5: Laya fine-tuning -- dataset build + RLCD training for the "same_entity" noul question.

Follows the recipe in NandhaKishorM/laya's own fine-tuning notebook
(notebooks/laya_finetune_typed_decisions_2xT4_kaggle.ipynb): gold-distribution soft targets,
a noisy-logit policy-gradient objective scored with `laya.common.proper_reward` (a strictly
proper scoring rule) plus a soft cross-entropy term, then post-hoc temperature calibration on a
held-out slice. That notebook fine-tunes on an existing HF benchmark dataset; this script builds
the *same* training-item shape from our own candidate pairs + ground truth instead.

NOT RUN HERE -- needs a GPU. This needs `pip install laya transformers datasets safetensors
huggingface_hub` (see requirements.txt) and network access to Hugging Face Hub for the base
checkpoints, both of which this authoring environment intentionally does not exercise. Verify on
the GPU machine: sanity-check a handful of built examples' `state`/`gold` (the --stage prepare
output), confirm `laya.common.build_sequence` accepts them without a length mismatch (the "markers
!= k" skip in build_training_item silently drops malformed items -- if a large fraction get
dropped, that's a bug worth investigating before spending GPU time on the rest).

Two checkpoints are fine-tuned separately, per the challenge's Router usage:
  - "english":       only on pairs where BOTH sides are Latin-script (its natural domain).
  - "multilingual":  on the FULL pair set (every script), so it also covers India's Devanagari/
                      Tamil business names -- the Router then sends non-Latin text there.
Train/calibration entities are drawn only from the S1 entities NOT held out for GBDT/ensemble
validation (src/common.py:stratified_split_by_s1's train_ids), and the calibration slice is a
further split of THAT train set (never the val set, and never a benchmark's own holdout) --
satisfying "held-out slice of your own train split" for the calibration-fitting step.

Run (from code/business_entity_resolution/), two stages:
    # 1. Build the jsonl datasets (fast, CPU-only, needs no GPU/laya import):
    python -m src.laya_finetune --stage prepare

    # 2. Fine-tune each role (GPU; single process or `torchrun --nproc_per_node=2` for 2xT4):
    python -m src.laya_finetune --stage train --role english
    torchrun --standalone --nproc_per_node=2 -m src.laya_finetune --stage train --role multilingual

Writes:
    data_processed/laya_{role}_{train,calib}.jsonl   (--stage prepare)
    models/laya_{role}/                               (--stage train: fine-tuned checkpoint dir)
"""
from __future__ import annotations

import argparse
import json
import os
import random
from pathlib import Path
from typing import Dict, List

from tqdm.auto import tqdm

from src import common
from src.features import load_entity_lookup

ROLES = ("english", "multilingual")
BASE_CHECKPOINTS = {"english": "convaiinnovations/laya", "multilingual": "convaiinnovations/laya-multilingual"}

SAME_ENTITY_INSTRUCTIONS = (
    "record_a and record_b each describe one business as \"name | address | country\", taken "
    "from two different, independently-collected data sources. Decide whether they refer to the "
    "SAME real-world business."
)
SAME_ENTITY_CRITERIA = {
    "true": (
        "The same business despite surface noise between the two sources: legal-suffix "
        "abbreviations (Corp/Corporation, Ltd/Limited, Pvt/Private, Inc, or French SARL/SAS/SASU/"
        "SA), typos, transliteration differences, word-order changes, punctuation differences, a "
        "DBA/trade name standing in for the same legal entity, a missing address component (no "
        "PIN/postal code, no state), or a landmark-based address reference (e.g. \"near SBI ATM\") "
        "for what is otherwise the same location."
    ),
    "false": (
        "Different businesses: distinct legal entities even when the names look similar (e.g. "
        "two different branches of the same chain at different addresses), a coincidental name "
        "match, or an address that is clearly a different location."
    ),
}


def record_text(rec: dict) -> str:
    return f"{rec.get('business_name', '')} | {rec.get('business_address', '')} | {rec.get('country', '')}"


def make_example(rec_a: dict, rec_b: dict, label_true: bool) -> dict:
    return {
        "state": {"record_a": record_text(rec_a), "record_b": record_text(rec_b)},
        "questions": {"same_entity": {"type": "noul", "instructions": SAME_ENTITY_INSTRUCTIONS,
                                       "criteria": SAME_ENTITY_CRITERIA}},
        # Hard 0/1 targets: our ground truth has no soft/teacher distribution to draw
        # probabilities from (unlike the notebook's RLCD-teacher-labeled benchmark), so
        # "probabilities" is a one-hot vector rather than a genuinely soft label. proper_reward
        # and the soft-CE term still work correctly on one-hot targets; it just means the
        # RLCD objective has less to teach about calibrated *uncertainty* than a soft-labeled
        # dataset would, only about calibrated *confidence* via the post-hoc temperature fit.
        "gold": {"same_entity": {
            "label": "true" if label_true else "false",
            "probabilities": {"true": 1.0 if label_true else 0.0, "false": 0.0 if label_true else 1.0},
        }},
        "meta": {"script_a": rec_a.get("name_script", "other"), "script_b": rec_b.get("name_script", "other")},
    }


def build_examples(candidate_map: Dict[str, set], lookup: Dict[str, dict], truth: Dict[str, set],
                    s1_ids: set, max_negatives_per_entity: int, seed: int) -> List[dict]:
    """Every positive pair, plus up to `max_negatives_per_entity` randomly-chosen hard negatives
    per S1 entity. Using all ~K negatives per entity on the full training set produces millions of
    examples -- more than a Kaggle session can tokenize in RAM, let alone train on in time -- and
    heavily over-weights "false" anyway."""
    rng = random.Random(seed)
    examples = []
    for s1 in tqdm(sorted(s1_ids), desc="laya examples", unit="entity", mininterval=2.0):
        s1_rec = lookup.get(s1)
        if s1_rec is None:
            continue
        true_ids = truth.get(s1, set())
        cands = sorted(c for c in candidate_map.get(s1, ()) if c in lookup)
        positives = [c for c in cands if c in true_ids]
        negatives = [c for c in cands if c not in true_ids]
        if len(negatives) > max_negatives_per_entity:
            negatives = rng.sample(negatives, max_negatives_per_entity)
        for cand, label_true in [(c, True) for c in positives] + [(c, False) for c in negatives]:
            cand_rec = lookup[cand]
            examples.append(make_example(s1_rec, cand_rec, label_true))
            examples.append(make_example(cand_rec, s1_rec, label_true))  # swapped-order copy
    return examples


def route_by_script(examples: List[dict]) -> Dict[str, List[dict]]:
    english = [e for e in examples if e["meta"]["script_a"] == "latin" and e["meta"]["script_b"] == "latin"]
    return {"english": english, "multilingual": list(examples)}  # multilingual sees everything


def write_jsonl(examples: List[dict], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for ex in examples:
            f.write(json.dumps({k: v for k, v in ex.items() if k != "meta"}, ensure_ascii=False) + "\n")


def read_jsonl(path: Path) -> List[dict]:
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


# ------------------------------------------------------------------------------------- prepare

def _cap(ids, limit: int, seed: int) -> set:
    ids = sorted(ids)
    if limit and len(ids) > limit:
        ids = random.Random(seed).sample(ids, limit)
    return set(ids)


def stage_prepare(repo_root: Path, candidates_path, val_frac: float, calib_frac: float, seed: int,
                  max_negatives_per_entity: int, max_finetune_entities: int = 6000,
                  max_calib_entities: int = 1000):
    """max_finetune_entities: the upstream recipe fine-tunes on ~6k items in minutes on 2xT4;
    6000 entities x (positives + 3 negatives) x 2 orderings is ~60-80k examples, roughly an hour
    per role. Raise it if you have GPU time to spare."""
    candidates_path = candidates_path or (common.data_processed_dir(repo_root) / "candidate_pairs_train.tsv")
    ground_truth = common.sampled_ground_truth(repo_root, candidates_path)
    truth = common.ground_truth_map(ground_truth)
    train_ids, val_ids = common.stratified_split_by_s1(ground_truth, val_frac=val_frac, seed=seed)
    print(f"[laya_finetune/prepare] {len(train_ids)} train-pool / {len(val_ids)} held-out-val S1 "
          f"entities (val entities are NEVER used for fine-tuning or calibration)")

    # Calibration slice carved out of the TRAIN pool only, with a different seed offset so it's
    # not the same split as train_gbdt's/ensemble's val split.
    train_gt = ground_truth[ground_truth["source1_entity_id"].isin(train_ids)]
    finetune_ids, calib_ids = common.stratified_split_by_s1(train_gt, val_frac=calib_frac, seed=seed + 1)
    finetune_ids = _cap(finetune_ids, max_finetune_entities, seed)
    calib_ids = _cap(calib_ids, max_calib_entities, seed + 1)
    print(f"[laya_finetune/prepare] using {len(finetune_ids)} fine-tune / {len(calib_ids)} calibration "
          f"S1 entities (capped)")

    candidate_map = common.read_id_list_tsv(candidates_path)
    chosen = finetune_ids | calib_ids
    needed_ids = set(chosen)
    for s1 in chosen:
        needed_ids.update(candidate_map.get(s1, ()))
    lookup = load_entity_lookup(repo_root, "train", needed_ids)

    finetune_examples = build_examples(candidate_map, lookup, truth, finetune_ids, max_negatives_per_entity, seed)
    calib_examples = build_examples(candidate_map, lookup, truth, calib_ids, max_negatives_per_entity, seed + 1)
    print(f"[laya_finetune/prepare] built {len(finetune_examples)} fine-tune / "
          f"{len(calib_examples)} calibration examples (each pair contributes 2: original + "
          f"swapped-order copy)")

    finetune_by_role = route_by_script(finetune_examples)
    calib_by_role = route_by_script(calib_examples)

    out_dir = common.data_processed_dir(repo_root)
    for role in ROLES:
        train_path = out_dir / f"laya_{role}_train.jsonl"
        calib_path = out_dir / f"laya_{role}_calib.jsonl"
        write_jsonl(finetune_by_role[role], train_path)
        write_jsonl(calib_by_role[role], calib_path)
        print(f"[laya_finetune/prepare] {role}: {len(finetune_by_role[role])} train / "
              f"{len(calib_by_role[role])} calib -> {train_path.name}, {calib_path.name}")


# --------------------------------------------------------------------------------------- train
#
# Mirrors laya's own fine-tuning notebook (see module docstring) closely enough to be a drop-in
# reproduction of its recipe, generalized to (a) read our jsonl datasets instead of the
# LocalLLaMA/typed-decisions HF dataset, (b) run as a plain script instead of notebook cells, and
# (c) work single-process (Colab, 1 GPU) as well as under `torchrun` DDP (Kaggle 2xT4) -- the
# notebook is DDP-only. Everything below this point needs torch/transformers/laya installed and a
# GPU; none of it is imported or run by --stage prepare.

def _build_training_item(tok, cfg, example: dict):
    from laya.common import build_sequence, render_options, QTYPES

    state = example["state"]
    q = example["questions"]["same_entity"]
    gold = example["gold"]["same_entity"]["probabilities"]
    target = [gold["false"], gold["true"]]
    label = int(target[1] > target[0])
    qdict = {"t": "noul", "ins": q["instructions"], "crit": q.get("criteria", {})}
    k = len(render_options(qdict))
    seq, markers = build_sequence(tok, state, qdict, cfg["max_len"], cfg["head_max_len"])
    if len(markers) != k:
        return None
    return {"ids": seq, "markers": markers, "qtype": QTYPES["noul"], "target": target, "label": label}


def _collate_train_batch(items, pad_id):
    import torch

    n, L = len(items), max(len(it["ids"]) for it in items)
    kmax = max(len(it["markers"]) for it in items)
    ids = torch.full((n, L), pad_id, dtype=torch.long)
    att = torch.zeros((n, L), dtype=torch.long)
    mpos = torch.zeros((n, kmax), dtype=torch.long)
    mmask = torch.zeros((n, kmax), dtype=torch.bool)
    target = torch.zeros((n, kmax), dtype=torch.float32)
    for i, it in enumerate(items):
        ids[i, :len(it["ids"])] = torch.tensor(it["ids"])
        att[i, :len(it["ids"])] = 1
        k = len(it["markers"])
        mpos[i, :k] = torch.tensor(it["markers"])
        mmask[i, :k] = True
        target[i, :len(it["target"])] = torch.tensor(it["target"], dtype=torch.float32)
    return {
        "input_ids": ids, "attention_mask": att, "marker_pos": mpos, "marker_mask": mmask,
        "target": target,
        "qtype": torch.tensor([it["qtype"] for it in items]),
        "label": torch.tensor([it["label"] for it in items]),
    }


def _fit_one_temp(sel):
    import torch

    if len(sel) < 10:
        return 1.0
    kmax = max(len(z) for z, _ in sel)
    Z = torch.full((len(sel), kmax), -1e4)
    T = torch.zeros((len(sel), kmax))
    for i, (z, t) in enumerate(sel):
        Z[i, :len(z)] = torch.tensor(z)
        T[i, :len(t)] = torch.tensor(t, dtype=torch.float32)
    log_t = torch.zeros(1, requires_grad=True)
    opt = torch.optim.LBFGS([log_t], lr=0.1, max_iter=100)

    def closure():
        opt.zero_grad()
        loss = -(T * torch.log_softmax(Z / log_t.exp(), -1)).sum(-1).mean()
        loss.backward()
        return loss

    opt.step(closure)
    return float(torch.clamp(log_t.exp(), 0.1, 10.0).item())


def stage_train(repo_root: Path, role: str, epochs: int, micro_batch: int, grad_accum: int,
                 lr_encoder: float, lr_head: float, calib_max: int, seed: int):
    import time
    import torch
    import torch.distributed as dist
    from torch.nn.parallel import DistributedDataParallel as DDP
    from safetensors.torch import load_file, save_file
    from transformers import AutoTokenizer
    from huggingface_hub import snapshot_download
    from laya.agent import _fix_tokenizer_config
    from laya.common import build_model, proper_reward, QTYPES

    ddp_mode = "WORLD_SIZE" in os.environ and int(os.environ.get("WORLD_SIZE", "1")) > 1
    if ddp_mode:
        dist.init_process_group("nccl")
        rank, world_size = dist.get_rank(), dist.get_world_size()
        local_rank = int(os.environ.get("LOCAL_RANK", "0"))
        torch.cuda.set_device(local_rank)
        device = torch.device("cuda", local_rank)
    else:
        rank, world_size, local_rank = 0, 1, 0
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        if device.type != "cuda" and rank == 0:
            print("[laya_finetune/train] WARNING: no CUDA device visible -- this will be very "
                  "slow or fail outright. This script must be run on the GPU machine, not here.")

    model_repo = BASE_CHECKPOINTS[role]
    model_dir = snapshot_download(model_repo)
    _fix_tokenizer_config(model_dir)
    tok = AutoTokenizer.from_pretrained(os.path.join(model_dir, "tokenizer"))
    with open(os.path.join(model_dir, "rl_agent_config.json")) as f:
        cfg = json.load(f)
    cfg["gradient_checkpointing"] = True
    cfg["max_tokens_per_batch"] = 4096

    data_dir = common.data_processed_dir(repo_root)
    prepared_by = "laya_finetune.py --stage prepare (notebook Section 5a)"
    train_examples = read_jsonl(common.require(data_dir / f"laya_{role}_train.jsonl", prepared_by))
    calib_examples = read_jsonl(common.require(data_dir / f"laya_{role}_calib.jsonl", prepared_by))
    if rank == 0:
        print(f"[laya_finetune/train] role={role} model={model_repo} device={device} "
              f"ddp={ddp_mode} world_size={world_size}")
        print(f"[laya_finetune/train] {len(train_examples)} train / {len(calib_examples)} "
              f"calibration examples (calibration held out of training, never seen by any rank)")

    quiet = rank != 0
    train_items = [it for ex in tqdm(train_examples, desc="tokenize train", disable=quiet, mininterval=2.0)
                   if (it := _build_training_item(tok, cfg, ex)) is not None]
    calib_items = [it for ex in tqdm(calib_examples, desc="tokenize calib", disable=quiet, mininterval=2.0)
                   if (it := _build_training_item(tok, cfg, ex)) is not None]
    dropped = (len(train_examples) - len(train_items)) + (len(calib_examples) - len(calib_items))
    if dropped and rank == 0:
        print(f"[laya_finetune/train] WARNING: {dropped} example(s) dropped by build_sequence "
              f"(marker/option count mismatch) -- inspect on the GPU machine if this is a large "
              f"fraction of the dataset.")

    if len(calib_items) > calib_max:
        random.Random(seed).shuffle(calib_items)
        calib_items = calib_items[:calib_max]

    my_items = train_items[rank::world_size]
    model = build_model(cfg, encoder_dir=os.path.join(model_dir, "encoder"))
    weights = load_file(os.path.join(model_dir, "model.safetensors"))
    model.load_state_dict(weights, strict=True)
    model.encoder.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.head_checkpointing = True
    model.to(device)
    model.train()

    ddp_model = DDP(model, device_ids=[local_rank], find_unused_parameters=True) if ddp_mode else model

    sigma_start, sigma_end = 0.4, 0.1
    enc_params = [p for n, p in ddp_model.named_parameters() if "encoder." in n]
    head_params = [p for n, p in ddp_model.named_parameters() if "encoder." not in n]
    optimizer = torch.optim.AdamW(
        [{"params": enc_params, "lr": lr_encoder}, {"params": head_params, "lr": lr_head}],
        weight_decay=0.01,
    )
    total_updates = max(1, (len(my_items) // (micro_batch * grad_accum)) * epochs)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_updates, eta_min=1e-6)
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")

    output_dir = common.models_dir(repo_root) / f"laya_{role}"
    if rank == 0:
        print(f"[laya_finetune/train] {len(train_items)} usable train items "
              f"({len(calib_items)} held out for calibration) | {len(my_items)} on this rank | "
              f"{epochs} epochs -> {output_dir}")
    t0 = time.time()

    for epoch in range(epochs):
        random.seed(seed + epoch + rank)
        random.shuffle(my_items)
        epoch_loss, n_batches, accum_step = 0.0, 0, 0
        optimizer.zero_grad(set_to_none=True)
        sigma = sigma_start + (sigma_end - sigma_start) * (epoch / max(1, epochs - 1))

        for b_idx in tqdm(range(0, len(my_items), micro_batch), desc=f"epoch {epoch + 1}/{epochs}",
                          unit="batch", disable=quiet, mininterval=5.0):
            chunk = my_items[b_idx:b_idx + micro_batch]
            if not chunk:
                continue
            batch = _collate_train_batch(chunk, tok.pad_token_id)
            amp_dtype = torch.float16 if device.type == "cuda" else torch.bfloat16
            with torch.autocast(device.type, dtype=amp_dtype, enabled=device.type in ("cuda", "cpu")):
                logits, act = ddp_model(
                    batch["input_ids"].to(device), batch["attention_mask"].to(device),
                    batch["marker_pos"].to(device), batch["marker_mask"].to(device),
                    batch["qtype"].to(device),
                )
            logits = logits.float()
            mask = batch["marker_mask"].to(device)
            k = mask.sum(-1, keepdim=True).float()
            target = batch["target"].to(device)

            group_size = 4
            eps = torch.randn((group_size,) + logits.shape, device=device) * sigma * mask
            eps = (eps - eps.sum(-1, keepdim=True) / k) * mask
            z = logits.detach().unsqueeze(0) + eps
            q = torch.softmax(z.masked_fill(~mask, -1e4), -1)
            with torch.no_grad():
                r = proper_reward(q, target.unsqueeze(0), batch["qtype"].to(device), mask, w_sph=0.75, w_rps=1.0)
                adv = r - r.mean(0, keepdim=True)
                adv = adv / (adv.std() + 1e-6)

            logp = -(((z - logits.unsqueeze(0)) ** 2) * mask).sum(-1) / (2 * sigma ** 2)
            loss_rl = -(adv * logp).mean()
            loss_ce = -(target * torch.log_softmax(logits.masked_fill(~mask, -1e4), -1)).sum(-1).mean()
            loss = (loss_rl + 1.0 * loss_ce) / grad_accum + 0.0 * act.sum()

            scaler.scale(loss).backward()
            accum_step += 1
            if accum_step % grad_accum == 0 or (b_idx + micro_batch) >= len(my_items):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(ddp_model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)

            epoch_loss += loss.item() * grad_accum
            n_batches += 1
            if rank == 0 and n_batches % 50 == 0:
                print(f"  epoch {epoch + 1}/{epochs} step {n_batches} loss={loss.item() * grad_accum:.4f} "
                      f"reward={r.mean().item():.3f} lr={scheduler.get_last_lr()[0]:.2e}")

        if rank == 0:
            print(f"=== epoch {epoch + 1}/{epochs} done in {time.time() - t0:.1f}s | "
                  f"avg loss {epoch_loss / max(1, n_batches):.4f} ===")
        if ddp_mode:
            dist.barrier()
        if rank == 0:
            ckpt_dir = output_dir / "checkpoint_latest"
            ckpt_dir.mkdir(parents=True, exist_ok=True)
            ckpt_sd = {k: v.half().contiguous().cpu() for k, v in model.state_dict().items()}
            save_file(ckpt_sd, str(ckpt_dir / "model.safetensors"))
            model.encoder.config.save_pretrained(str(ckpt_dir / "encoder"))
            tok.save_pretrained(str(ckpt_dir / "tokenizer"))
            with open(ckpt_dir / "checkpoint_meta.json", "w") as f:
                json.dump({"epoch": epoch + 1, "total_epochs": epochs,
                           "avg_loss": epoch_loss / max(1, n_batches)}, f, indent=2)

    if ddp_mode:
        dist.barrier()

    if rank == 0:
        print("\n[laya_finetune/train] fitting post-training calibration temperature on the "
              "held-out calibration slice (never trained on, above)...")
        del optimizer, scaler, scheduler
        if device.type == "cuda":
            torch.cuda.empty_cache()
        model.eval()
        calib_preds = []
        with torch.no_grad():
            for c_idx in range(0, len(calib_items), 16):
                c_chunk = calib_items[c_idx:c_idx + 16]
                if not c_chunk:
                    continue
                cb = _collate_train_batch(c_chunk, tok.pad_token_id)
                amp_dtype = torch.float16 if device.type == "cuda" else torch.bfloat16
                with torch.autocast(device.type, dtype=amp_dtype, enabled=device.type in ("cuda", "cpu")):
                    l_sub, _ = model(
                        cb["input_ids"].to(device), cb["attention_mask"].to(device),
                        cb["marker_pos"].to(device), cb["marker_mask"].to(device),
                        cb["qtype"].to(device),
                    )
                l_np = l_sub.float().cpu().numpy()
                for r_idx, it in enumerate(c_chunk):
                    k = len(it["markers"])
                    calib_preds.append((it["qtype"], l_np[r_idx, :k], it["target"]))

        fitted_temps = [1.2, 1.2, 1.2]  # (choice, score, noul) -- QTYPES order; only "noul" is used here
        try:
            sel = [(z, t) for qtype, z, t in calib_preds if qtype == QTYPES["noul"]]
            if sel:
                fitted_temps[QTYPES["noul"]] = _fit_one_temp(sel)
            print(f"[laya_finetune/train] fitted noul temperature: {fitted_temps[QTYPES['noul']]:.3f} "
                  f"(on {len(sel)} held-out calibration items)")
        except Exception as exc:
            print(f"[laya_finetune/train] temperature fitting failed, keeping default 1.2: {exc}")

        output_dir.mkdir(parents=True, exist_ok=True)
        sd = {k: v.half().contiguous().cpu() for k, v in model.state_dict().items()}
        save_file(sd, str(output_dir / "model.safetensors"))
        model.encoder.config.save_pretrained(str(output_dir / "encoder"))
        tok.save_pretrained(str(output_dir / "tokenizer"))
        cfg["fine_tuned"] = True
        cfg["model_name"] = f"laya-entity-resolution-{role}"
        cfg["temperature"] = fitted_temps
        cfg.pop("temperature_by_options", None)
        with open(output_dir / "rl_agent_config.json", "w") as f:
            json.dump(cfg, f, indent=2)
        print(f"[laya_finetune/train] saved fine-tuned {role} checkpoint to {output_dir}")

    if ddp_mode:
        dist.destroy_process_group()


# ---------------------------------------------------------------------------------- Router glue

def build_routers(repo_root: Path) -> list:
    """One Router per visible GPU (both T4s on Kaggle's T4x2), each pinned to its own device, so
    ensemble.score_with_laya can run them in parallel. Falls back to a single CPU router. Prints
    where each checkpoint actually landed: laya silently moves a checkpoint to CPU if it doesn't
    fit in GPU memory, which would otherwise show up only as "CPU busy, GPU idle"."""
    import torch

    n_gpus = torch.cuda.device_count()
    devices = [f"cuda:{i}" for i in range(n_gpus)] or ["cpu"]
    routers = []
    for device in devices:
        router = build_router(repo_root, device=device)
        for name, agent in router._agents.items():
            print(f"[laya] router for {device}: checkpoint '{name}' loaded on {agent.device}")
            if device != "cpu" and agent.device.type != "cuda":
                print(f"[laya] WARNING: '{name}' fell back to CPU on {device} (likely GPU out of memory) "
                      f"-- scoring will be very slow. Free GPU memory (restart the kernel after fine-tuning) and retry.")
        routers.append(router)
    return routers


def build_router(repo_root: Path, device: str = None):
    """Assembles both fine-tuned checkpoints into a laya.Router, exactly the mechanism the
    challenge's fine-tuning step asks for: 'Use the Router (English vs multilingual checkpoint)
    so India records with Devanagari/Tamil business names route to the multilingual checkpoint.'
    `Router(models=...)` accepts local directories directly (laya/router.py: `Agent(repo, ...)`
    where `repo` is whatever string was passed -- a local dir short-circuits the Hub download).
    Falls back to the base (non-fine-tuned) checkpoints with a warning if a fine-tuned dir is
    missing, so ensemble.py/predict.py can still run end-to-end before fine-tuning is done."""
    from laya import Router

    models = {}
    for role in ROLES:
        local_dir = common.models_dir(repo_root) / f"laya_{role}"
        if local_dir.exists():
            models[role] = str(local_dir)
        else:
            print(f"[laya_finetune] WARNING: no fine-tuned '{role}' checkpoint at {local_dir}; "
                  f"falling back to the base '{BASE_CHECKPOINTS[role]}' checkpoint (zero-shot).")
            models[role] = BASE_CHECKPOINTS[role]
    return Router(models=models, device=device, preload=True)


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    common.add_repo_root_arg(parser)
    parser.add_argument("--stage", choices=["prepare", "train"], required=True)
    # prepare
    parser.add_argument("--candidates", type=str, default=None,
                         help="[prepare] Path to the train candidate_pairs.tsv. "
                              "Default: data_processed/candidate_pairs_train.tsv")
    parser.add_argument("--val-frac", type=float, default=0.2,
                         help="[prepare] Must match train_gbdt.py's --val-frac so 'val' means "
                              "the same held-out entities everywhere in the pipeline.")
    parser.add_argument("--calib-frac", type=float, default=0.15,
                         help="[prepare] Fraction of the TRAIN pool (not val) carved out for "
                              "Laya's post-training calibration fit.")
    parser.add_argument("--max-finetune-entities", type=int, default=6000,
                         help="[prepare] S1 entities used for fine-tuning examples (0 = all).")
    parser.add_argument("--max-negatives-per-entity", type=int, default=3,
                         help="[prepare] Hard negatives kept per S1 entity (all positives are kept).")
    # train
    parser.add_argument("--role", choices=list(ROLES), default=None,
                         help="[train] Which checkpoint to fine-tune.")
    parser.add_argument("--epochs", type=int, default=4)
    parser.add_argument("--micro-batch", type=int, default=8)
    parser.add_argument("--grad-accum", type=int, default=4)
    parser.add_argument("--lr-encoder", type=float, default=2.5e-5)
    parser.add_argument("--lr-head", type=float, default=1.0e-4)
    parser.add_argument("--calib-max", type=int, default=400)
    parser.add_argument("--seed", type=int, default=42)
    args = parser.parse_args()

    if args.stage == "prepare":
        stage_prepare(args.repo_root, args.candidates, args.val_frac, args.calib_frac, args.seed,
                      args.max_negatives_per_entity, args.max_finetune_entities)
    else:
        if not args.role:
            raise SystemExit("--stage train requires --role {english,multilingual}")
        stage_train(args.repo_root, args.role, args.epochs, args.micro_batch, args.grad_accum,
                    args.lr_encoder, args.lr_head, args.calib_max, args.seed)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile $PIPELINE_DIR/src/ensemble.py
"""Step 6: ensemble -- score a shortlist of candidates with fine-tuned Laya, stack with the GBDT
features, refit the F_0.5 threshold on the stack's own output, and report macro F_0.5 by country
and singleton status.

Laya scores a SHORTLIST, not every candidate pair: with ~30 candidates per S1 entity the full
pair set is millions of rows, and a transformer forward pass per pair doesn't fit in a Kaggle
session. The shortlist is each S1 entity's top LAYA_TOP_N candidates by GBDT probability, restricted
to those with gbdt_prob >= LAYA_MIN_GBDT_PROB. Pairs below that bar are already confident GBDT
rejections; Laya's job is the semantic cases near the top of the list (DBA names, domain-as-name,
paraphrased legal names). Unscored pairs get laya_prob=0 and laya_scored=0, so the stacker can tell
"Laya said no" from "Laya never looked". The SAME shortlist rule is applied in predict.py.

Run (from code/business_entity_resolution/), after train_gbdt.py and laya_finetune.py --stage train:
    python -m src.ensemble --features ../../data_processed/features_train.parquet

Writes:
    models/ensemble_lr.joblib, models/ensemble_gbdt_alt.txt, models/ensemble_stack_columns.json,
    models/ensemble_threshold.json, models/ensemble_config.json (shortlist settings),
    data_processed/ensemble_val_report.json
"""
from __future__ import annotations

import argparse
import json
from pathlib import Path
from typing import Dict, List

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from src import common
from src import train_gbdt
from src.features import load_entity_table, pair_entity_ids
from src.laya_finetune import SAME_ENTITY_CRITERIA, SAME_ENTITY_INSTRUCTIONS, build_routers

DEFAULT_THRESHOLDS = [round(t, 2) for t in np.arange(0.05, 0.96, 0.05)]
LAYA_TOP_N = 3
LAYA_MIN_GBDT_PROB = 0.05
LAYA_CHUNK = 20_000
# Records are short ("name | address | country"), so a large batch keeps a T4 busy; 64 left it
# mostly idle waiting on CPU-side tokenization between tiny forward passes.
LAYA_BATCH_SIZE = 256


# ------------------------------------------------------------------------------------- Laya pass

def laya_shortlist_mask(features_df: pd.DataFrame, gbdt_prob: np.ndarray, top_n: int,
                        min_prob: float) -> np.ndarray:
    frame = pd.DataFrame({"s1": features_df["source1_entity_id"].to_numpy(), "p": gbdt_prob})
    rank = frame.groupby("s1")["p"].rank(method="first", ascending=False)
    return ((rank <= top_n) & (frame["p"] >= min_prob)).to_numpy()


def record_texts(table: pd.DataFrame) -> pd.Series:
    """Same "name | address | country" format laya_finetune.record_text uses for training."""
    return table["business_name"] + " | " + table["business_address"] + " | " + table["country"]


def score_with_laya(routers, features_df: pd.DataFrame, table: pd.DataFrame, mask: np.ndarray,
                    batch_size: int = LAYA_BATCH_SIZE, chunk_size: int = LAYA_CHUNK) -> np.ndarray:
    """P(same_entity) for rows where `mask` is True (0.0 elsewhere). Chunks are spread across
    `routers` (one per GPU, from laya_finetune.build_routers) by a thread pool -- each worker
    checks a router out of a queue, so no two chunks share a GPU at once. torch releases the GIL
    during forward passes, so two T4s genuinely run concurrently."""
    import queue
    from concurrent.futures import ThreadPoolExecutor

    if not isinstance(routers, (list, tuple)):
        routers = [routers]
    questions = {"same_entity": {"type": "noul", "instructions": SAME_ENTITY_INSTRUCTIONS,
                                 "criteria": SAME_ENTITY_CRITERIA}}
    texts = record_texts(table)
    out = np.zeros(len(features_df), dtype=np.float32)
    idx = np.flatnonzero(mask)
    s1_col = features_df["source1_entity_id"].to_numpy()
    cand_col = features_df["candidate_entity_id"].to_numpy()
    chunks = [idx[start:start + chunk_size] for start in range(0, len(idx), chunk_size)]
    print(f"[laya] scoring {len(idx)} shortlisted pairs of {len(features_df)} on {len(routers)} device(s)")

    free = queue.Queue()
    for router in routers:
        free.put(router)

    def _score(rows):
        router = free.get()
        try:
            a = texts.loc[s1_col[rows]].tolist()
            b = texts.loc[cand_col[rows]].tolist()
            requests = [{"state": {"record_a": x, "record_b": y}, "questions": questions} for x, y in zip(a, b)]
            results = router.predict_batch(requests, batch_size=batch_size)
            return rows, [res["answers"]["same_entity"]["noul"] for res in results]
        finally:
            free.put(router)

    with ThreadPoolExecutor(max_workers=len(routers)) as pool, \
            tqdm(total=len(idx), desc="laya scoring", unit="pair", unit_scale=True) as bar:
        for rows, probs in pool.map(_score, chunks):
            out[rows] = probs
            bar.update(len(rows))
    return out


# ------------------------------------------------------------------------------- stacking table

def build_stack_frame(features_df: pd.DataFrame, gbdt_prob: np.ndarray, laya_prob: np.ndarray,
                      laya_scored: np.ndarray, base_cols: List[str]) -> pd.DataFrame:
    stack = features_df[base_cols].astype("float32").copy()
    stack["gbdt_prob"] = gbdt_prob.astype("float32")
    stack["laya_prob"] = laya_prob.astype("float32")
    stack["laya_scored"] = laya_scored.astype("float32")
    return stack


def stack_columns(base_cols: List[str]) -> List[str]:
    return list(base_cols) + ["gbdt_prob", "laya_prob", "laya_scored"]


# ---------------------------------------------------------------------------------- train stack

def train_logistic_stacker(X_train, y_train):
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler

    pipe = Pipeline([
        ("scale", StandardScaler()),
        ("lr", LogisticRegression(max_iter=2000, class_weight="balanced")),
    ])
    pipe.fit(X_train, y_train)
    return pipe


def train_shallow_gbdt_stacker(X_train, y_train, X_val, y_val, cols: List[str], seed: int = 42):
    import lightgbm as lgb

    train_set = lgb.Dataset(X_train, label=y_train, feature_name=cols)
    val_set = lgb.Dataset(X_val, label=y_val, feature_name=cols, reference=train_set)
    params = {
        "objective": "binary", "metric": "binary_logloss", "learning_rate": 0.05,
        "num_leaves": 7, "max_depth": 3, "min_data_in_leaf": 30,
        "is_unbalance": True, "verbose": -1, "seed": seed,
    }
    return lgb.train(params, train_set, num_boost_round=500, valid_sets=[val_set],
                     callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])


def score_stack(model, kind: str, X) -> np.ndarray:
    if kind == "logistic":
        return model.predict_proba(X)[:, 1]
    return model.predict(X, num_iteration=getattr(model, "best_iteration", None))


# ---------------------------------------------------------------------- threshold + evaluation

def predicted_sets(pairs_df: pd.DataFrame, scores: np.ndarray, threshold: float,
                   eval_ids) -> Dict[str, set]:
    pred = {s1: set() for s1 in eval_ids}
    keep = scores >= threshold
    for s1, cand in zip(pairs_df["source1_entity_id"].to_numpy()[keep],
                        pairs_df["candidate_entity_id"].to_numpy()[keep]):
        if s1 in pred:
            pred[s1].add(cand)
    return pred


def best_threshold(pairs_df: pd.DataFrame, scores: np.ndarray, truth: Dict[str, set],
                   eval_ids: set, thresholds=None) -> Dict:
    thresholds = thresholds if thresholds is not None else DEFAULT_THRESHOLDS
    eval_truth = {s1: ids for s1, ids in truth.items() if s1 in eval_ids}
    best = {"threshold": 0.5, "macro_f0_5": -1.0}
    sweep = {}
    for t in tqdm(thresholds, desc="threshold sweep", unit="t"):
        pred = predicted_sets(pairs_df, scores, t, eval_ids)
        precision, recall = common.precision_recall_macro(pred, eval_truth)
        macro_f05 = common.macro_f_beta(pred, eval_truth, beta=0.5)
        sweep[str(t)] = {"precision": precision, "recall": recall, "macro_f0_5": macro_f05}
        if macro_f05 > best["macro_f0_5"]:
            best = {"threshold": t, "macro_f0_5": macro_f05, "precision": precision, "recall": recall}
    return {"best": best, "sweep": sweep}


def breakdown_report(pairs_df: pd.DataFrame, scores: np.ndarray, threshold: float,
                     truth: Dict[str, set], eval_ids: set, table: pd.DataFrame) -> Dict:
    """Macro F_0.5 by country (esp. France, which Laya never trains on) and by singleton status."""
    pred = predicted_sets(pairs_df, scores, threshold, eval_ids)
    eval_truth = {s1: ids for s1, ids in truth.items() if s1 in eval_ids}
    per_entity = common.f_beta_per_entity(pred, eval_truth, beta=0.5)
    countries = table["country"]

    by_country: Dict[str, List[float]] = {}
    singleton_scores, nonsingleton_scores = [], []
    for s1, score in per_entity.items():
        country = (countries.get(s1) or "unknown") if s1 in countries.index else "unknown"
        by_country.setdefault(country, []).append(score)
        (singleton_scores if not eval_truth.get(s1) else nonsingleton_scores).append(score)

    def _mean(v):
        return sum(v) / len(v) if v else float("nan")

    return {
        "overall_macro_f0_5": _mean(list(per_entity.values())),
        "by_country": {c: {"n": len(v), "macro_f0_5": _mean(v)} for c, v in sorted(by_country.items())},
        "singletons": {"n": len(singleton_scores), "macro_f0_5": _mean(singleton_scores)},
        "non_singletons": {"n": len(nonsingleton_scores), "macro_f0_5": _mean(nonsingleton_scores)},
    }


# --------------------------------------------------------------------------------------- driver

def run(repo_root: Path, features_path, val_frac: float, seed: int, laya_batch_size: int,
        laya_top_n: int, laya_min_gbdt_prob: float, max_entities: int = 60_000):
    """The stacker trains ONLY on the GBDT's held-out val entities (split in half: stack-train /
    stack-eval). On the GBDT's own training entities gbdt_prob is overfit -- the model has seen
    those labels -- so a stacker fit there learns to over-trust it. Laya fine-tuning also only
    used train-pool entities, so neither input is overfit on these rows. `max_entities` caps how
    many val entities are used (Laya scoring is the slow part)."""
    import joblib
    import lightgbm as lgb

    features_path = features_path or (common.data_processed_dir(repo_root) / "features_train.parquet")
    features_df = pd.read_parquet(common.require(features_path, "features.py (notebook Section 3)"))
    ground_truth = common.sampled_ground_truth(repo_root)
    truth = common.ground_truth_map(ground_truth)

    # Same split as train_gbdt.py, so these are entities the GBDT never trained on.
    _, gbdt_val_ids = common.stratified_split_by_s1(ground_truth, val_frac=val_frac, seed=seed)
    val_gt = ground_truth[ground_truth["source1_entity_id"].isin(gbdt_val_ids)]
    if max_entities and len(val_gt) > max_entities:
        val_gt = val_gt.sample(n=max_entities, random_state=seed)
    stack_train_ids, stack_eval_ids = common.stratified_split_by_s1(val_gt, val_frac=0.5, seed=seed + 2)
    print(f"[ensemble] stacking on GBDT-held-out entities only: {len(stack_train_ids)} stack-train / "
          f"{len(stack_eval_ids)} stack-eval S1 entities")

    features_df = features_df[features_df["source1_entity_id"].isin(stack_train_ids | stack_eval_ids)]
    features_df = features_df.reset_index(drop=True)
    labels = train_gbdt.label_pairs(features_df, ground_truth).astype(int).to_numpy()
    train_mask = features_df["source1_entity_id"].isin(stack_train_ids).to_numpy()
    eval_mask = features_df["source1_entity_id"].isin(stack_eval_ids).to_numpy()

    models_path = common.models_dir(repo_root)
    with open(common.require(models_path / "gbdt_feature_columns.json", "train_gbdt.py (notebook Section 4)")) as f:
        gbdt_cols = json.load(f)
    gbdt_model = lgb.Booster(model_file=str(models_path / "gbdt_model.txt"))
    gbdt_prob = train_gbdt.score_with_gbdt(gbdt_model, features_df, cols=gbdt_cols)

    table = load_entity_table(repo_root, "train", ids=pair_entity_ids(features_df))
    shortlist = laya_shortlist_mask(features_df, gbdt_prob, laya_top_n, laya_min_gbdt_prob)
    routers = build_routers(repo_root)
    laya_prob = score_with_laya(routers, features_df, table, shortlist, batch_size=laya_batch_size)
    del routers

    stack_df = build_stack_frame(features_df, gbdt_prob, laya_prob, shortlist, gbdt_cols)
    cols = stack_columns(gbdt_cols)
    X_train, y_train = stack_df[train_mask][cols], labels[train_mask]
    X_eval, y_eval = stack_df[eval_mask][cols], labels[eval_mask]
    print(f"[ensemble] {len(cols)} stacking columns: {X_train.shape[0]} stack-train / {X_eval.shape[0]} stack-eval rows")

    lr_model = train_logistic_stacker(X_train, y_train)
    gbdt_alt_model = train_shallow_gbdt_stacker(X_train, y_train, X_eval, y_eval, cols, seed=seed)

    pairs_eval = features_df.loc[eval_mask, ["source1_entity_id", "candidate_entity_id"]].reset_index(drop=True)
    gbdt_only = best_threshold(pairs_eval, gbdt_prob[eval_mask], truth, stack_eval_ids)
    results = {"gbdt_only_baseline": {
        "threshold_sweep": gbdt_only,
        "breakdown": breakdown_report(pairs_eval, gbdt_prob[eval_mask], gbdt_only["best"]["threshold"],
                                      truth, stack_eval_ids, table)}}
    print(f"[ensemble] GBDT-only baseline: best threshold={gbdt_only['best']['threshold']} "
          f"macro_F0.5={gbdt_only['best']['macro_f0_5']:.4f}")

    for name, model, kind in (("logistic", lr_model, "logistic"), ("gbdt_alt", gbdt_alt_model, "gbdt")):
        eval_scores = score_stack(model, kind, X_eval)
        threshold_report = best_threshold(pairs_eval, eval_scores, truth, stack_eval_ids)
        breakdown = breakdown_report(pairs_eval, eval_scores, threshold_report["best"]["threshold"],
                                     truth, stack_eval_ids, table)
        results[name] = {"threshold_sweep": threshold_report, "breakdown": breakdown}
        print(f"[ensemble] {name}: best threshold={threshold_report['best']['threshold']} "
              f"macro_F0.5={threshold_report['best']['macro_f0_5']:.4f}")
        print(json.dumps(breakdown, indent=2))

    joblib.dump(lr_model, models_path / "ensemble_lr.joblib")
    gbdt_alt_model.save_model(str(models_path / "ensemble_gbdt_alt.txt"),
                              num_iteration=gbdt_alt_model.best_iteration)
    with open(models_path / "ensemble_stack_columns.json", "w") as f:
        json.dump(cols, f, indent=2)
    with open(models_path / "ensemble_threshold.json", "w") as f:
        json.dump({name: results[name]["threshold_sweep"]["best"]["threshold"]
                   for name in ("gbdt_only_baseline", "logistic", "gbdt_alt")}, f, indent=2)
    with open(models_path / "ensemble_config.json", "w") as f:
        json.dump({"laya_top_n": laya_top_n, "laya_min_gbdt_prob": laya_min_gbdt_prob}, f, indent=2)
    report_path = common.data_processed_dir(repo_root) / "ensemble_val_report.json"
    with open(report_path, "w") as f:
        json.dump(results, f, indent=2)
    print(f"[ensemble] wrote models + {report_path}. Compare 'logistic'/'gbdt_alt' against "
          f"'gbdt_only_baseline' before trusting that Laya helped.")


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    common.add_repo_root_arg(parser)
    parser.add_argument("--features", type=str, default=None,
                        help="Path to features_train.parquet. Default: data_processed/features_train.parquet")
    parser.add_argument("--val-frac", type=float, default=0.2)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--laya-batch-size", type=int, default=LAYA_BATCH_SIZE)
    parser.add_argument("--laya-top-n", type=int, default=LAYA_TOP_N)
    parser.add_argument("--laya-min-gbdt-prob", type=float, default=LAYA_MIN_GBDT_PROB)
    parser.add_argument("--max-entities", type=int, default=60_000,
                        help="Cap on GBDT-held-out S1 entities used for stacking (0 = all).")
    args = parser.parse_args()
    run(args.repo_root, args.features, args.val_frac, args.seed, args.laya_batch_size,
        args.laya_top_n, args.laya_min_gbdt_prob, args.max_entities)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile $PIPELINE_DIR/src/predict.py
"""Step 7: full test-set inference -- blocking -> features -> GBDT -> Laya (shortlist) -> ensemble
-> threshold. Produces both submission files.

Run LAST, after train_gbdt.py, laya_finetune.py (prepare + train for both roles) and ensemble.py.

Run (from code/business_entity_resolution/):
    python -m src.predict --stack-model logistic

Writes (paths match what utils/validate_submission.py expects, run from student_resource/):
    output/candidate_pairs.tsv     -- the test-set candidates actually fed to the final model
    output/matching_results.tsv    -- final matches
"""
from __future__ import annotations

import argparse
import json
from pathlib import Path

from src import blocking, common, ensemble, features, train_gbdt
from src.laya_finetune import build_routers


def run(repo_root: Path, k: int, stack_model: str, laya_batch_size: int,
        threshold_override: float):
    import lightgbm as lgb

    models_path = common.models_dir(repo_root)

    # ---- 1. blocking ----
    s1_df, s2_df, s3_df = blocking.load_blocking_inputs(repo_root, "test")
    all_s1_ids = list(s1_df["entity_id"])
    print(f"[predict] blocking: {len(s1_df)} S1 test entities (all of them), k={k}")
    candidate_map = blocking.generate_candidates(s1_df, s2_df, s3_df, k=k)
    del s2_df, s3_df
    candidate_path = common.output_dir(repo_root) / "candidate_pairs.tsv"
    common.write_id_list_tsv(candidate_map, candidate_path, "source1_entity_id", "candidate_entity_ids")
    print(f"[predict] wrote {candidate_path} ({sum(len(v) for v in candidate_map.values())} pairs)")

    # ---- 2. features ----
    pairs_df = features.build_pairs_frame(candidate_map)
    del candidate_map
    table = features.load_entity_table(repo_root, "test", ids=features.pair_entity_ids(pairs_df))
    features_df = features.compute_features(pairs_df, table)
    del pairs_df

    if features_df.empty:
        print("[predict] WARNING: zero candidate pairs survived blocking -- every S1 test entity will "
              "be predicted as a singleton. This almost certainly means a blocking bug.")
        _write_matches(repo_root, {s1: set() for s1 in all_s1_ids})
        return

    # ---- 3. GBDT ----
    with open(common.require(models_path / "gbdt_feature_columns.json", "train_gbdt.py (notebook Section 4)")) as f:
        gbdt_cols = json.load(f)
    gbdt_model = lgb.Booster(model_file=str(models_path / "gbdt_model.txt"))
    gbdt_prob = train_gbdt.score_with_gbdt(gbdt_model, features_df, cols=gbdt_cols)
    print(f"[predict] GBDT scored {len(gbdt_prob)} pairs (mean prob {gbdt_prob.mean():.4f})")

    # ---- 4+5. Laya shortlist + ensemble stack, or GBDT alone ----
    if stack_model == "gbdt_only":
        # Chosen when the ensemble didn't beat the GBDT-only baseline on stack-eval: skips Laya
        # (and all its GPU time) entirely.
        final_prob = gbdt_prob
    else:
        with open(common.require(models_path / "ensemble_config.json", "ensemble.py (notebook Section 6)")) as f:
            cfg = json.load(f)
        shortlist = ensemble.laya_shortlist_mask(features_df, gbdt_prob, cfg["laya_top_n"], cfg["laya_min_gbdt_prob"])
        routers = build_routers(repo_root)
        laya_prob = ensemble.score_with_laya(routers, features_df, table, shortlist, batch_size=laya_batch_size)
        del routers
        with open(models_path / "ensemble_stack_columns.json") as f:
            stack_cols = json.load(f)
        stack_df = ensemble.build_stack_frame(features_df, gbdt_prob, laya_prob, shortlist, gbdt_cols)[stack_cols]
        if stack_model == "logistic":
            import joblib
            final_prob = ensemble.score_stack(joblib.load(models_path / "ensemble_lr.joblib"), "logistic", stack_df)
        else:
            alt_model = lgb.Booster(model_file=str(models_path / "ensemble_gbdt_alt.txt"))
            final_prob = ensemble.score_stack(alt_model, "gbdt", stack_df)
    print(f"[predict] {stack_model} mean prob {final_prob.mean():.4f}")

    # ---- 6. threshold ----
    if threshold_override is not None:
        threshold = threshold_override
    else:
        with open(common.require(models_path / "ensemble_threshold.json", "ensemble.py (notebook Section 6)")) as f:
            thresholds = json.load(f)
        threshold = thresholds["gbdt_only_baseline" if stack_model == "gbdt_only" else stack_model]
    print(f"[predict] decision threshold: {threshold}")

    # ---- 7. matching_results.tsv (every S1 seeded first -> one row each; subset of candidates) ----
    matches = ensemble.predicted_sets(features_df, final_prob, threshold, all_s1_ids)
    _write_matches(repo_root, matches)


def _write_matches(repo_root: Path, matches):
    matching_path = common.output_dir(repo_root) / "matching_results.tsv"
    common.write_id_list_tsv(matches, matching_path, "source1_entity_id", "matched_entity_ids")
    n_matched = sum(1 for ids in matches.values() if ids)
    print(f"[predict] wrote {matching_path}: {len(matches)} S1 entities, {n_matched} with >=1 match, "
          f"{len(matches) - n_matched} predicted singletons")


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    common.add_repo_root_arg(parser)
    parser.add_argument("--k", type=int, default=blocking.DEFAULT_K)
    parser.add_argument("--stack-model", choices=["logistic", "gbdt_alt", "gbdt_only"], default="logistic")
    parser.add_argument("--laya-batch-size", type=int, default=ensemble.LAYA_BATCH_SIZE)
    parser.add_argument("--threshold", type=float, default=None,
                        help="Override the threshold saved by ensemble.py.")
    args = parser.parse_args()
    run(args.repo_root, args.k, args.stack_model, args.laya_batch_size, args.threshold)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile $STUDENT_RESOURCE_DIR/utils/validate_submission.py
#!/usr/bin/env python3
"""
ML Challenge 2026 — Submission Validator

Run this BEFORE submitting. It checks your output files against every formatting
rule the scorer enforces, so you can catch a rejection locally instead of burning
a submission. It reads only your output files and the test source files (to learn
which S1 entities are required and which S2/S3 IDs exist); it never needs the
ground truth and never computes your score.

It validates two files:

* ``matching_results.tsv`` (required) — your final matches, the file scored on the
  leaderboard.
* ``candidate_pairs.tsv`` (optional) — the candidate set from your blocking stage.
  When present, the validator also checks that your final matches are a subset of
  your candidates and *warns* (never fails) otherwise. When absent it is skipped
  with a warning; it is still expected in your final submission zip.

Stdlib only, Python 3.8+. Run from the ``student_resource/`` directory::

    python3 utils/validate_submission.py \
        --matching output/matching_results.tsv \
        --candidate output/candidate_pairs.tsv \
        --test-dir dataset/test

Exit code 0 means the files are safe to submit; 1 means fix the listed issues
(warnings never fail the run).

ID-existence check (off by default). By default the validator does NOT check that
every matched/candidate ID actually exists in the test set: that check loads all
Source-2/3 IDs into memory, which on the full ~1.7M-entity test set costs a few GB
(more when ``candidate_pairs.tsv`` is included). The default run therefore stays fast
and light and verifies every other rule; it prints a warning noting the check was
skipped. Pass ``--check-ids`` to turn it on (it reads ``test_source2.tsv`` /
``test_source3.tsv`` from ``--test-dir``); a missing/garbage matched ID only lowers
your score rather than being rejected by the scorer, so this check is a diagnostic,
not a gate. If ``--check-ids`` runs out of memory, drop ``--candidate`` (the candidate
cross-check is the biggest memory user, and the matching file is the only one scored).
"""

import argparse
import os
import sys

DELIM = "\t"
MAX_EXAMPLES = 5  # how many offending IDs to show per issue
MATCHING_HEADER = ["source1_entity_id", "matched_entity_ids"]
CANDIDATE_HEADER = ["source1_entity_id", "candidate_entity_ids"]


def read_ids(path):
    """Return the set of first-column entity IDs from a source TSV.

    The header row is skipped and blank lines are ignored.
    """
    with open(path, encoding="utf-8") as f:
        next(f, None)  # skip header
        return {line.split(DELIM, 1)[0].strip() for line in f if line.strip()}


def examples(items):
    """Return a short, human-readable sample of ``items`` for an error message."""
    items = sorted(items)
    shown = ", ".join(items[:MAX_EXAMPLES])
    if len(items) > MAX_EXAMPLES:
        return f"{len(items)} total, e.g. {shown}, ..."
    return shown


def load_match_targets(test_dir, warnings):
    """Return the set of valid S2/S3 match IDs, or ``None`` if unavailable.

    Only called when ``--check-ids`` is on. When ``test_source2.tsv`` or
    ``test_source3.tsv`` is missing we cannot check that matched IDs exist, so we
    record a warning and return ``None`` to signal that the existence check should be
    skipped.
    """
    targets = set()
    for name in ("test_source2.tsv", "test_source3.tsv"):
        path = os.path.join(test_dir, name)
        if not os.path.isfile(path):
            warnings.append(
                f"{path} not found — skipping the (optional) check that matched "
                f"IDs exist in the test set. Every other rule is still checked. "
                f"This is the lighter-memory mode; provide test_source2/3.tsv to "
                f"enable the ID-existence check."
            )
            return None
        targets |= read_ids(path)
    return targets


def validate_id_list_file(path, expected_header, col_label, required, valid_ids, errors):
    """Validate one results-style TSV (matching or candidate).

    Applies the shared formatting rules and appends any problems to ``errors``.
    Returns a ``{source1_id: set(matched/candidate ids)}`` mapping, or ``None`` on a
    fatal problem (missing file, empty file, or a broken header) that stops parsing.
    """
    if not os.path.isfile(path):
        errors.append(f"File not found: {path}")
        return None

    name = os.path.basename(path)
    mapping = {}
    seen, dup_rows, intra_dupes = set(), set(), set()
    self_matches, wrong_prefix, unknown = set(), set(), set()
    n_rows = empties = 0

    with open(path, encoding="utf-8") as f:
        header = f.readline()
        if not header:
            errors.append(f"{name} is empty.")
            return None
        if DELIM not in header and "," in header:  # the #1 mistake: a CSV
            errors.append(
                f"{name}: header has no TAB but contains commas — the file looks "
                "COMMA-separated. Submissions must be TAB-separated (.tsv); "
                "write it with df.to_csv(sep='\\t', index=False)."
            )
            return None
        cols = [c.strip().lower() for c in header.rstrip("\n").split(DELIM)]
        if cols != expected_header:
            errors.append(
                f"{name}: unexpected header {cols}. "
                f"Expected exactly {expected_header} (tab-separated)."
            )
            return None

        for line_num, line in enumerate(f, start=2):
            s1, tab, rest = line.partition(DELIM)
            if not tab:
                if s1.strip():
                    errors.append(
                        f"{name}: malformed row (no tab) at line {line_num}: "
                        f"{line.rstrip()!r}"
                    )
                continue

            n_rows += 1
            if s1 in seen:
                dup_rows.add(s1)
            seen.add(s1)

            ids = rest.rstrip("\n").split(",") if rest.strip() else []
            if not ids:
                empties += 1
                mapping[s1] = set()
                continue
            if len(ids) != len(set(ids)):
                intra_dupes.add(s1)
            id_set = set(ids)
            mapping[s1] = id_set
            for mid in id_set:
                if mid.startswith("S1-"):
                    self_matches.add(mid)
                elif not mid.startswith(("S2-", "S3-")):
                    wrong_prefix.add(mid)
                elif valid_ids is not None and mid not in valid_ids:
                    unknown.add(mid)

    # Aggregate the per-category findings. Each entry is (offenders, message);
    # only non-empty categories become errors.
    findings = [
        (
            dup_rows,
            "{name}: duplicate source1_entity_id row(s): {ex}. "
            "Each S1 entity may appear on only one row.",
        ),
        (
            intra_dupes,
            "{name}: repeated ID inside a {col} list for: {ex}. "
            "No duplicate IDs are allowed within a list.",
        ),
        (
            self_matches,
            "{name}: {col} contains Source-1 IDs (self-matches): {ex}. "
            "Only S2-/S3- IDs are allowed.",
        ),
        (
            wrong_prefix,
            "{name}: {col} contains IDs without an S2-/S3- prefix: {ex}.",
        ),
        (
            unknown,
            "{name}: {col} references IDs not in the test "
            "Source-2/3 files: {ex}.",
        ),
        (
            required - seen,
            "{name}: required S1 entity(ies) missing: {ex}. "
            "Every entity in test_source1.tsv needs a row (empty = no match).",
        ),
        (
            seen - required,
            "{name}: row(s) using an S1 ID that is not in the test set: {ex}.",
        ),
    ]
    for offenders, message in findings:
        if offenders:
            errors.append(message.format(name=name, ex=examples(offenders), col=col_label))

    print(f"  {name}: {n_rows} rows ({empties} empty, {n_rows - empties} non-empty).")
    return mapping


def validate(matching_path, candidate_path, test_dir, check_ids=False):
    """Validate the submission output(s); return ``(errors, warnings)`` lists.

    ``check_ids`` (``--check-ids``) turns on the optional, memory-heavy check that
    every matched/candidate ID exists in the test Source-2/3 files. It is off by
    default so the common run stays fast and light.
    """
    errors, warnings = [], []

    source1 = os.path.join(test_dir, "test_source1.tsv")
    if not os.path.isfile(source1):
        errors.append(f"Test source1 file not found: {source1} (check --test-dir).")
        return errors, warnings
    required = read_ids(source1)
    print(f"  required S1 entities: {len(required)}")

    if check_ids:
        valid_ids = load_match_targets(test_dir, warnings)
        if valid_ids is not None:
            print(f"  valid S2/S3 match IDs: {len(valid_ids)}")
    else:
        valid_ids = None
        warnings.append(
            "ID-existence check is OFF (the default) — not checking that matched/"
            "candidate IDs exist in the test set. Every other rule is still checked. "
            "Re-run with --check-ids to enable it (needs test_source2/3.tsv; uses "
            "more memory). A nonexistent ID only lowers your score, never rejects "
            "your submission."
        )

    matched = validate_id_list_file(
        matching_path, MATCHING_HEADER, "matched_entity_ids", required, valid_ids, errors
    )

    # candidate_pairs.tsv is optional: if it's absent we skip its checks with a
    # warning (it's still expected in your final submission zip). A missing
    # candidate file never fails this run on its own.
    candidate = None
    if candidate_path and os.path.isfile(candidate_path):
        candidate = validate_id_list_file(
            candidate_path, CANDIDATE_HEADER, "candidate_entity_ids",
            required, valid_ids, errors,
        )
    elif candidate_path:
        warnings.append(
            f"{candidate_path} not found — skipping candidate_pairs.tsv checks. "
            "It is optional here, but your final submission zip must include "
            "output/candidate_pairs.tsv."
        )

    # Soft check: your final matches should come from your blocking candidates.
    # A matched ID absent from candidate_pairs.tsv usually means a pipeline bug,
    # so we warn but never fail on it.
    if matched is not None and candidate is not None:
        offenders = {
            s1 for s1, mids in matched.items() if mids - candidate.get(s1, set())
        }
        if offenders:
            warnings.append(
                f"{len(offenders)} S1 entity(ies) have matched IDs not present in "
                f"candidate_pairs.tsv, e.g. {examples(offenders)}. Final matches "
                "normally come from your blocking candidates — double-check these."
            )

    return errors, warnings


def main():
    parser = argparse.ArgumentParser(
        description="Validate ML Challenge 2026 submission output files before submitting."
    )
    parser.add_argument(
        "--matching",
        "-m",
        default="output/matching_results.tsv",
        help="Path to matching_results.tsv (default: %(default)s)",
    )
    parser.add_argument(
        "--candidate",
        "-c",
        default=None,
        help="Path to candidate_pairs.tsv "
        "(default: output/candidate_pairs.tsv if it exists).",
    )
    parser.add_argument(
        "--test-dir",
        "-t",
        default="dataset/test",
        help="Folder with test_source1/2/3.tsv (default: %(default)s). "
        "test_source2/3.tsv are only read when --check-ids is given.",
    )
    parser.add_argument(
        "--check-ids",
        action="store_true",
        help="Also check that every matched/candidate ID exists in the test "
        "Source-2/3 files. Off by default (loads all S2/S3 IDs into memory — a few "
        "GB on the full test set). A nonexistent ID only lowers your score, so this "
        "is a diagnostic, not a submission gate.",
    )
    args = parser.parse_args()

    # candidate_pairs.tsv is optional; default to the conventional path and let
    # validate() skip (with a warning) if the file isn't there.
    candidate_path = args.candidate or "output/candidate_pairs.tsv"

    print("ML Challenge 2026 — submission validator")
    print(f"  test dir: {args.test_dir}")
    try:
        errors, warnings = validate(
            args.matching, candidate_path, args.test_dir, check_ids=args.check_ids
        )
    except UnicodeDecodeError:
        print()
        print("FAIL — 1 issue(s) to fix before submitting:")
        print(
            f"  1. A file is not valid UTF-8 text (most likely {args.matching} or "
            f"{candidate_path}). Re-save it as a plain UTF-8, tab-separated .tsv — "
            "not cp1252/Latin-1, and not a compressed or binary file (.gz/.xlsx/"
            ".parquet) renamed to .tsv. In pandas: "
            "df.to_csv(path, sep='\\t', index=False, encoding='utf-8')."
        )
        return 1
    except OSError as exc:
        print()
        print("FAIL — 1 issue(s) to fix before submitting:")
        print(f"  1. Could not read a file: {exc}.")
        return 1

    print()
    for warning in warnings:
        print(f"WARNING: {warning}")
    if errors:
        print(f"FAIL — {len(errors)} issue(s) to fix before submitting:")
        for i, error in enumerate(errors, 1):
            print(f"  {i}. {error}")
        return 1
    print("PASS — no blocking issues found. Safe to submit.")
    return 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile $STUDENT_RESOURCE_DIR/Documentation_template.md
# ML Challenge 2026: Business Entity Resolution Solution Template

**Team Name:** [Your Team Name]  
**Team Members:** [List all team members]  
**Submission Date:** [Date]

---

## 1. Executive Summary
*Provide a brief 2-3 sentence overview of your approach and key innovations.*

---

## 2. Methodology

### 2.1 Problem Analysis
*Key insights discovered during EDA — noise patterns, address variations, missing fields, etc.*

### 2.2 Solution Strategy
*Outline your high-level approach.*

**Approach Type:** [Blocking + Classifier / End-to-End / Graph-Based / Hybrid, etc]  
**Core Innovation:** [Brief description of your main technical contribution]

---

## 3. Candidate Generation (Blocking)
*Describe how you reduced the comparison space to a manageable candidate set.*

- **Blocking keys used:** [e.g., PIN code, phonetic name encoding, TF-IDF, etc.]
- **Candidate pairs generated:** [total]
- **How you ensured true matches were not lost:**

---

## 4. Matching Model

**Features used:**
- Name features: [e.g., Jaccard, Levenshtein, phonetic encoding]
- Address features: [e.g., token overlap, edit distance, PIN code matching]
- Other: []

**Model type:** [e.g., XGBoost, Siamese Network, Transformer, etc.]  
**Threshold selection method:** [e.g., F_0.5 optimization on validation set]

---

## 5. Laya Integration

### 5.1 Why Laya, and where it sits in the pipeline

Section 4's GBDT scores every blocked candidate pair on hand-engineered string-similarity
features: normalized Levenshtein and Jaro-Winkler distance, word- and character-n-gram TF-IDF
cosine, and address/pin-code/city overlap (see `features.py`). Those features are all functions
of surface string form, which makes the GBDT blind to pairs where two records describe the same
business but don't look alike as strings: a DBA record where the legal name and trade name are
unrelated strings (`Ectolumdrex dba X+ Madison Inc`), a record where the business name is a
domain rather than a name (`wilfordhancock.com`), or a legal name that's been reordered or
paraphrased between sources. To catch these, we run a second, heterogeneous scorer in parallel
over the same blocking output: **Laya** (`NandhaKishorM/laya`), a small (~322M–421M parameter),
Apache-2.0-licensed, non-autoregressive text classifier, fine-tuned to answer the matching
question directly in natural language rather than through engineered features. Because it reads
the two records as text and reasons about the question itself, it can recognize the DBA relation,
the domain-as-name case, or a paraphrased legal name as the same semantic content even when the
GBDT's string metrics see two dissimilar strings. Laya does not replace the GBDT; its output is
folded back into the GBDT's feature table as one more signal (Section 5.5).

### 5.2 How a candidate pair is scored

Every candidate pair is scored with a single `noul` (yes/no) question, `same_entity`, built from
the pair's two normalized records (`src/laya_finetune.py`):

```python
questions = {
    "same_entity": {
        "type": "noul",
        "instructions": (
            "record_a and record_b each describe one business as \"name | address | country\", "
            "taken from two different, independently-collected data sources. Decide whether "
            "they refer to the SAME real-world business."
        ),
        "criteria": {
            "true": "The same business despite surface noise between the two sources: legal-suffix "
                    "abbreviations, typos, transliteration differences, word-order changes, a "
                    "DBA/trade name standing in for the same legal entity, a missing address "
                    "component, or a landmark-based address reference for the same location.",
            "false": "Different businesses: distinct legal entities even when the names look "
                     "similar, a coincidental name match, or a clearly different address.",
        },
    }
}
state = {"record_a": "<name> | <address> | <country>", "record_b": "<name> | <address> | <country>"}
```

`same_entity`'s `noul` answer is `P(true)`, i.e. the model's probability that the two records
refer to the same business. Rather than scoring pairs one at a time, `ensemble.score_with_laya`
builds this state/question pair for every row of the candidate feature table and hands the whole
batch to `Router.predict_batch(requests, batch_size=...)`, which groups requests by checkpoint and
shares forward passes across them — the entire `candidate_pairs.tsv` set for a split is scored in
this one batched call rather than per-pair inference calls.

### 5.3 Fine-tuning approach

The Laya checkpoints (`convaiinnovations/laya`, `convaiinnovations/laya-multilingual`) are
general-purpose System-1 decision models, not trained on this task's records or vocabulary; the
project's working assumption is that they need fine-tuning to be useful here rather than being
applied zero-shot (this has not been measured, since nothing in this pipeline has been executed —
see the caveat in Section 5.6). Fine-tuning data is built entirely from our own pipeline's
intermediate output, not any external dataset: `laya_finetune.build_examples` pairs each S1
entity's blocking candidates (`candidate_pairs_train.tsv`) with the training ground truth
(`train_ground_truth.tsv`, via `common.ground_truth_map`). A pair is a **positive** example when
the candidate is in that S1 entity's `matched_entity_ids`, and a **negative** ("hard negative")
example when it survived blocking but is not a true match — i.e. it's a plausible-looking
non-match rather than a random, easily-distinguished pair, which is what makes it a useful
training signal. Every pair is duplicated with `record_a`/`record_b` swapped
(`make_example(rec_a, rec_b, ...)` and `make_example(cand_rec, s1_rec, ...)`) so the model isn't
implicitly trained to expect the S1 record first.

Training follows Laya's own RLCD (reinforcement learning against strictly proper scoring rules)
fine-tuning recipe, reproduced from the upstream repository's fine-tuning notebook: a noisy-logit
policy-gradient objective scored with `laya.common.proper_reward`, combined with a soft
cross-entropy term, optimized over several epochs (`stage_train` in `laya_finetune.py`). After
training, calibration temperatures are refit with `_fit_one_temp` on a calibration slice carved
out of **our own** training split — specifically, a further stratified split of the S1 entities
not held out for GBDT/ensemble validation (`--calib-frac`, disjoint from both the val split and
from any of Laya's own benchmark defaults) — so the model's reported confidence is calibrated to
this task's label distribution, not a general-purpose benchmark's.

### 5.4 Multilingual routing

Training data is split by script before fine-tuning (`route_by_script` in `laya_finetune.py`):
the `english` checkpoint is fine-tuned only on pairs where both records' business names are
Latin-script, while the `multilingual` checkpoint is fine-tuned on the full pair set, including
India's Devanagari- and Tamil-script business names (and any mixed-script record, since a record
that's part Latin and part Devanagari/Tamil doesn't route cleanly to either checkpoint alone at
train time). At inference, `build_router` assembles both fine-tuned checkpoints into a
`laya.Router`, which detects each request's script/language and dispatches non-English text to
the `multilingual` checkpoint automatically — this is how mixed-script Indian business names are
routed without any script-handling logic of our own beyond selecting which checkpoint each
training example belongs to.

### 5.5 Combining with the GBDT (stacking, not replacement)

Laya is not trusted standalone as the final decision-maker for a precision-heavy metric like
F_0.5: `noul` questions are known to be sensitive to how the criteria are phrased and can be
biased toward one answer on ambiguous inputs, and the `multilingual` checkpoint's fine-tuning data
never includes a France record (Section 5.4's routing is script-based, and France's business
names are Latin-script, so they train the `english` checkpoint — France is simply never a
fine-tuning example for the model that would otherwise be expected to generalize best to it). For
both reasons, Laya's output is stacked with the GBDT rather than substituted for it
(`ensemble.py`): `score_with_laya` produces `laya_prob` for every candidate pair, and
`build_stack_frame` appends it — alongside the GBDT's own `gbdt_prob` — as two extra columns onto
the GBDT's original feature table. A new meta-classifier is then trained on this combined table:
`train_logistic_stacker` (a scaled `LogisticRegression`, the primary/interpretable option) and
`train_shallow_gbdt_stacker` (a shallow LightGBM model, `num_leaves=7`, `max_depth=3`, kept as a
comparison alternative). Finally, `best_threshold` sweeps the decision threshold on the
**ensemble's own** output specifically, rather than reusing the plain GBDT's threshold from
Section 4 — the two models' probability distributions aren't the same, so their F_0.5-optimal
cutoffs aren't expected to match.

### 5.6 Evaluation discipline

Before trusting the Laya-augmented ensemble over the GBDT-only baseline, the two must be compared
on the same held-out validation split: `ensemble.py`'s `breakdown_report` computes macro F_0.5 on
that split broken down by country — with particular scrutiny on France, since (Section 5.5) no
France record ever appears in Laya's fine-tuning or calibration data, so its behavior there is
pure zero-shot generalization — and separately for singleton entities specifically, since under
F_0.5 correctly predicting "no match" is worth full credit (1.0) and a false merge on a singleton
scores 0.0. **As of this write-up, that comparison has not been run**: this entire integration was
authored and only statically checked (no GPU was available in the authoring environment), so
`ensemble_val_report.json` (Laya + stacking) has not yet been compared against
`gbdt_val_report.json` (GBDT-only). This section documents the comparison the pipeline is built
to make, not a result — the actual comparison, and whichever of `logistic`/`gbdt_alt` wins it, is
outstanding work before this can be called an improvement rather than an untested hypothesis.

### 5.7 Fair-play note

Laya runs entirely locally on our own fine-tuned checkpoint — there is no call to a hosted
inference API, and no external data source is consulted at inference or fine-tuning time. The
base checkpoints (`convaiinnovations/laya`, 421M parameters; `convaiinnovations/laya-multilingual`,
322M parameters) are Apache-2.0 licensed and both well under the competition's 8B-parameter cap.
Fine-tuning data is derived exclusively from `train_ground_truth.tsv` and this pipeline's own
blocking output (`candidate_pairs_train.tsv`) — no business registry, geocoding service, or other
external lookup is used anywhere in this integration, keeping it inside the challenge's
prohibition on external data lookup.

---

## 6. Results & Error Analysis

- **F_0.5 Score (macro):** [your best validation score]
- **Common false positives (wrong merges):** [brief description]
- **Common false negatives (missed matches):** [brief description]

---

## 7. Conclusion
*Summarize your approach, key achievements, and lessons learned in 2-3 sentences.*

---

## Appendix

### A. Code Artefacts
*Your complete, runnable code ships in the submission zip under
`code/business_entity_resolution/` (all source in `src/`, with a `README.md` and
`requirements.txt`). Summarise its structure and the entry point(s) to reproduce
`output/matching_results.tsv` and `output/candidate_pairs.tsv` here.*

### B. Additional Results
*Include any additional charts, graphs, or detailed results.*

---

**Note:** Teams can modify sections according to their approach while maintaining clarity and technical depth.


In [ ]:
# Verify every file above actually landed on disk with current content, instead of trusting that
# each %%writefile cell ran. A %%writefile cell errors and halts "Run All" on some inputs (e.g. a
# genuinely empty file used to do this); every cell after the failure is then silently skipped,
# leaving old file content in place with no error of its own -- that's what caused a later,
# confusing "module 'src.common' has no attribute ..." several sections down, not a code bug.
import hashlib
import py_compile

_expected = {}
_expected[f"{PIPELINE_DIR}/requirements.txt"] = "878874ac57ddbb89cdfc37c850016ec337caa5f887937c414d238bb26304ecce"
_expected[f"{PIPELINE_DIR}/README.md"] = "ec470d456836c7d999a4b44e301fa5d49ce8b41b2744cfd54e89b5ac019b6af0"
_expected[f"{PIPELINE_DIR}/src/__init__.py"] = "02fc1ee85c512fe4ea48627c575e82e709ef4581f32ba7b2f06115ac91641b9f"
_expected[f"{PIPELINE_DIR}/src/common.py"] = "0d7db8fd9c354b9b6e2ad54d8c12bbac840ae90a729da9f89d82e1aefc90f5ff"
_expected[f"{PIPELINE_DIR}/src/normalize.py"] = "133ac6f036f9e1be8819c9976a6c046eff9a5c3a9f2582f6dcaae411067ffc29"
_expected[f"{PIPELINE_DIR}/src/blocking.py"] = "7470e645cef1a92f834d6c6c4f0988f5f0af327459e37b51c890cb3fd68b1ea3"
_expected[f"{PIPELINE_DIR}/src/features.py"] = "aeba641ceb9c2956df2ac079f66990b84e0361b690ee945d6d276b3d10cec0cd"
_expected[f"{PIPELINE_DIR}/src/train_gbdt.py"] = "6b8f813b515fed6e8c9a12fc2355b9614e7fea9cf0d4b2eeba8bbad0d7252431"
_expected[f"{PIPELINE_DIR}/src/laya_finetune.py"] = "98cdfa3d5e15451fe4e9f76f43c828b61a58b8a50177e858bf04e887f1403867"
_expected[f"{PIPELINE_DIR}/src/ensemble.py"] = "e78b9602cb48e88a7e855896bd6faba870f2519d2eae7cafbf6ab940e9363375"
_expected[f"{PIPELINE_DIR}/src/predict.py"] = "fdf477e65ac7ba317db46dcb328c416d81785d0125516839a668b670f13f939d"
_expected[f"{STUDENT_RESOURCE_DIR}/utils/validate_submission.py"] = "f96f59934383a15095914f507620c078c474e8f9c60e864b2d051ed173a22dfc"
_expected[f"{STUDENT_RESOURCE_DIR}/Documentation_template.md"] = "01361a4113d592a461826658b4209897bbd285347f7135900a3381c96c064586"

problems = []
for path, expected_hash in _expected.items():
    if not os.path.isfile(path):
        problems.append(f"{path}: missing -- its %%writefile cell above didn't run")
        continue
    actual = hashlib.sha256(open(path, "rb").read()).hexdigest()
    if actual != expected_hash:
        problems.append(f"{path}: on-disk content doesn't match this notebook -- re-run its %%writefile cell above")
    if path.endswith(".py"):
        try:
            py_compile.compile(path, doraise=True)
        except py_compile.PyCompileError as e:
            problems.append(f"{path}: does not compile -- {e}")

if problems:
    raise RuntimeError(
        "Section 0.2 did not finish writing the pipeline correctly:\n  " + "\n  ".join(problems)
        + "\n\nRe-run the listed %%writefile cell(s) above (scroll up), confirm each one's output "
          "starts with 'Writing' or 'Overwriting', then re-run this cell."
    )
print(f"Verified {len(_expected)} files on disk match this notebook.")


### 0.3 Install dependencies

Installs exactly `requirements.txt` (written above) and prints it first. Cell also self-heals a
numpy/pandas binary mismatch if the install ever causes one.


In [ ]:
%cd $PIPELINE_DIR
print(open("requirements.txt").read())
!pip install -q -r requirements.txt

import subprocess as _subprocess
try:
    import pandas as pd
    import numpy as np
except ValueError as e:
    print(f"numpy/pandas ABI mismatch detected ({e}) -- reinstalling both together...")
    _subprocess.run(["pip", "install", "-q", "--upgrade", "--force-reinstall", "numpy", "pandas"], check=True)
    import pandas as pd
    import numpy as np

import laya, sparse_dot_topn, torch, transformers
print("laya", laya.__version__, "| torch", torch.__version__, "| transformers", transformers.__version__,
      "| numpy", np.__version__, "| pandas", pd.__version__, "| CUDA available:", torch.cuda.is_available())


### 0.4 Provide the dataset

Attach **[prathameshfuke/dataset-ml](https://www.kaggle.com/datasets/prathameshfuke/dataset-ml)**
(private: be signed into the `prathameshfuke` Kaggle account) via **+ Add Data** in the right
sidebar. The next cell finds `train_source1.tsv` / `test_source1.tsv` anywhere under
`/kaggle/input/` and symlinks their folders into `student_resource/dataset/`. The fallback cells
after it are only for running outside Kaggle.


In [ ]:
import glob
import shutil


def find_dataset_under(root):
    train_hits = sorted(glob.glob(f"{root}/**/train_source1.tsv", recursive=True))
    test_hits = sorted(glob.glob(f"{root}/**/test_source1.tsv", recursive=True))
    for name, hits in (("train_source1.tsv", train_hits), ("test_source1.tsv", test_hits)):
        if len(hits) > 1:
            print(f"WARNING: multiple {name} found, using {hits[0]}:\n  " + "\n  ".join(hits))
    return (os.path.dirname(train_hits[0]) if train_hits else None,
            os.path.dirname(test_hits[0]) if test_hits else None)


def link_dataset(train_dir, test_dir):
    os.makedirs(f"{STUDENT_RESOURCE_DIR}/dataset", exist_ok=True)
    for name, src in (("train", train_dir), ("test", test_dir)):
        dst = f"{STUDENT_RESOURCE_DIR}/dataset/{name}"
        if os.path.islink(dst):
            os.remove(dst)
        elif os.path.isdir(dst):
            shutil.rmtree(dst)
        os.symlink(src, dst)


train_dir, test_dir = find_dataset_under("/kaggle/input") if os.path.isdir("/kaggle/input") else (None, None)
if train_dir and test_dir:
    link_dataset(train_dir, test_dir)
    print(f"train -> {train_dir}\ntest  -> {test_dir}\nsymlinked into {STUDENT_RESOURCE_DIR}/dataset/")
else:
    print("No train_source1.tsv/test_source1.tsv under /kaggle/input.")
    print("On Kaggle: + Add Data -> search 'dataset-ml' -> add prathameshfuke/dataset-ml, then re-run this cell.")


Fallbacks, only if the cell above found nothing (e.g. on Colab).

In [ ]:
# Fallback 1 -- a direct download URL to a zip with train/ and test/ at its root.
RUN_FALLBACK_URL = False
DATASET_URL = ""

if RUN_FALLBACK_URL:
    assert DATASET_URL, "Set DATASET_URL first."
    os.makedirs(f"{STUDENT_RESOURCE_DIR}/dataset", exist_ok=True)
    zip_path = f"{WORKDIR}/dataset_download.zip"
    subprocess.run(["curl", "-fL", "-o", zip_path, DATASET_URL], check=True)
    subprocess.run(["unzip", "-q", "-o", zip_path, "-d", f"{STUDENT_RESOURCE_DIR}/dataset"], check=True)


In [ ]:
# Fallback 2 -- Google Drive (Colab): folder containing train/ and test/.
RUN_FALLBACK_DRIVE = False
DRIVE_DATASET_DIR = "/content/drive/MyDrive/azlaya-dataset"

if RUN_FALLBACK_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    link_dataset(f"{DRIVE_DATASET_DIR}/train", f"{DRIVE_DATASET_DIR}/test")


In [ ]:
required = [
    "dataset/train/train_source1.tsv", "dataset/train/train_source2.tsv",
    "dataset/train/train_source3.tsv", "dataset/train/train_ground_truth.tsv",
    "dataset/test/test_source1.tsv", "dataset/test/test_source2.tsv", "dataset/test/test_source3.tsv",
]
missing = [f for f in required if not os.path.isfile(f"{STUDENT_RESOURCE_DIR}/{f}")]
if missing:
    raise FileNotFoundError("Missing dataset files -- run a dataset cell above first:\n  " + "\n  ".join(missing))
print("All dataset files found.")


### 0.5 Load the pipeline modules

Steps run **in this kernel** (not as `!python` subprocesses) so progress bars are live. Re-running
this cell reloads `src/` from disk.


In [ ]:
import gc
import importlib
import sys
from pathlib import Path

if PIPELINE_DIR not in sys.path:
    sys.path.insert(0, PIPELINE_DIR)

_PIPELINE_MODULE_NAMES = ["common", "features", "blocking", "train_gbdt", "laya_finetune", "ensemble", "normalize", "predict"]


def reload_pipeline():
    """Re-sync every pipeline module against what is CURRENTLY on disk in src/. Call this
    (every step cell below does, automatically) before using any pipeline function.

    Why this exists: a %%writefile cell only writes a file to disk -- it does not update code
    already loaded into this running kernel. normalize.py's own `from src import common`, for
    example, bound a specific `common` module OBJECT the first time normalize.py was imported;
    rewriting common.py on disk afterwards does not change that object. importlib.reload()
    re-executes a module's code INTO its existing object (unlike deleting it from sys.modules
    and re-importing, which creates a new object other modules keep no reference to), so every
    already-imported module sees the update immediately. This is what fixes
    "module 'src.common' has no attribute 'require'"-style errors after editing a %%writefile
    cell without a full kernel restart.

    If reload() itself ever errors (rare -- can happen after a large structural edit, e.g.
    removing a module-level name other code still imports by name), the reliable fallback is
    Kaggle's Run menu -> Restart Session, then Run All from the top.
    """
    mods = {}
    for name in _PIPELINE_MODULE_NAMES:
        full = f"src.{name}"
        mods[name] = importlib.reload(sys.modules[full]) if full in sys.modules else importlib.import_module(full)
    return mods


globals().update(reload_pipeline())

ROOT = Path(STUDENT_RESOURCE_DIR)
DP = ROOT / "data_processed"


def free_memory():
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except ImportError:
        pass


def run_streaming(cmd):
    """Run a subprocess (only torchrun) and stream its output live."""
    env = {**os.environ, "PYTHONUNBUFFERED": "1"}
    proc = subprocess.Popen(cmd, cwd=PIPELINE_DIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for chunk in iter(lambda: os.read(proc.stdout.fileno(), 4096), b""):
        sys.stdout.write(chunk.decode(errors="replace"))
        sys.stdout.flush()
    if proc.wait() != 0:
        raise RuntimeError(f"{' '.join(cmd)} exited with code {proc.returncode} -- see output above")


print("modules loaded | CPU cores:", os.cpu_count())


## 1. Normalize

Cleans names and addresses in all six source files (legal suffixes, DBA splitting, pin/postal
code, city guess, script detection), in parallel across every CPU core.

Set `SKIP_EXISTING = True` to reuse normalized files from a previous session. Only do this if
they were written by the current `normalize.py`; otherwise leave it `False`.


In [ ]:
reload_pipeline()
SKIP_EXISTING = False
normalize.run(ROOT, ["train", "test"], skip_existing=SKIP_EXISTING)
free_memory()


In [ ]:
import pandas as pd
sample = pd.read_csv(DP / "train_source1_normalized.tsv", sep="\t", dtype=str, nrows=10, keep_default_na=False)
sample[["business_name", "name_norm", "has_dba", "trade_name_norm", "address_norm", "pin_code", "city_guess", "name_script"]]


## 2. Blocking (candidate generation)

For each S1 entity, finds up to K=30 similar S2/S3 records in the same country. It combines a
name view (character 4-gram TF-IDF) and an address view (word TF-IDF, which carries pin, city and
street), searched with a multithreaded sparse top-n matrix product.

**Training uses a sample of `MAX_TRAIN_S1` S1 entities** (default 300k of the ~2.2M). Every later
training step uses the same sample. Their candidates are still searched against the full S2/S3
pool, so hard negatives stay realistic. The test set is always blocked in full (Section 7).

**Read the recall report:** `macro_entity_recall` is the ceiling for everything downstream.


In [ ]:
reload_pipeline()
MAX_TRAIN_S1 = 300_000
blocking.run(ROOT, "train", k=30, out_path=DP / "candidate_pairs_train.tsv", report_recall=True,
             val_frac=0.2, seed=42, max_s1=MAX_TRAIN_S1)
free_memory()


## 3. Feature engineering

Levenshtein, Jaro-Winkler, word/char TF-IDF cosine, address similarity, pin/city/country match,
DBA-aware name match. Computed in chunks for every candidate pair.


In [ ]:
reload_pipeline()
features.run(ROOT, "train", DP / "candidate_pairs_train.tsv", None)
free_memory()
feat = pd.read_parquet(DP / "features_train.parquet")
print(feat.shape)
feat.head()


## 4. GBDT matcher

LightGBM on the feature table, split by whole S1 entity so no validation entity leaks into
training. Reports precision / recall / macro F_0.5 on the held-out entities.


In [ ]:
import json
reload_pipeline()
_ = train_gbdt.train(ROOT, DP / "features_train.parquet", val_frac=0.2, seed=42,
                     num_boost_round=2000, early_stopping_rounds=50)
free_memory()
print(json.dumps(json.load(open(DP / "gbdt_val_report.json")), indent=2))


## 5. Laya fine-tuning

Two checkpoints fine-tuned with Laya's RLCD recipe: `english` on Latin-script pairs,
`multilingual` on all pairs (covers Devanagari/Tamil). Examples come from `MAX_FINETUNE_ENTITIES`
training entities: all their true matches, plus up to 3 hard negatives each, in both orderings.
The default of 6000 entities takes roughly an hour per checkpoint on T4 x2; raise it if you have
GPU time.

### 5a. Build the fine-tuning datasets (CPU)


In [ ]:
reload_pipeline()
MAX_FINETUNE_ENTITIES = 6000
laya_finetune.stage_prepare(ROOT, None, val_frac=0.2, calib_frac=0.15, seed=42,
                            max_negatives_per_entity=3, max_finetune_entities=MAX_FINETUNE_ENTITIES)
free_memory()
with open(DP / "laya_english_train.jsonl") as f:
    print(json.dumps(json.loads(next(f)), indent=2))


### 5b. Fine-tune both checkpoints

With 2+ GPUs this runs `torchrun` DDP across all of them, streaming output live. DDP needs
separate processes, so this step can't run in-kernel. With one GPU it runs in-kernel. Each
role saves a rolling `models/laya_<role>/checkpoint_latest/` after every epoch.

Fine-tuning uses the `%%writefile` version of `src/laya_finetune.py` on disk, so re-run its cell
in 0.2 first if you edited it.


In [ ]:
import torch

reload_pipeline()
N_GPUS = torch.cuda.device_count()
print(f"visible GPUs: {N_GPUS}", [torch.cuda.get_device_name(i) for i in range(N_GPUS)])
if N_GPUS == 0:
    raise RuntimeError("No GPU visible -- turn on an accelerator (Kaggle: Settings -> Accelerator) before Section 5.")

for role in ("english", "multilingual"):
    print(f"\n========== fine-tuning {role} ==========")
    if N_GPUS >= 2:
        run_streaming(["torchrun", "--standalone", f"--nproc_per_node={N_GPUS}",
                       "-m", "src.laya_finetune", "--stage", "train", "--role", role])
    else:
        laya_finetune.stage_train(ROOT, role, epochs=4, micro_batch=8, grad_accum=4,
                                  lr_encoder=2.5e-5, lr_head=1.0e-4, calib_max=400, seed=42)
    free_memory()


## 6. Ensemble

Trains the stacker **only on entities the GBDT never saw** (its held-out split, capped at
`MAX_STACK_ENTITIES`, halved into stack-train / stack-eval). On the GBDT's own training entities
its probabilities are overfit, and a stacker trained there would over-trust them.

Laya scores a shortlist: each entity's top 3 GBDT candidates with probability >= 0.05. The
stacker combines the GBDT features, `gbdt_prob` and `laya_prob`, then the F_0.5 threshold is
re-fit. The report compares logistic / shallow-GBDT stackers against a **GBDT-only baseline** on
the same entities, broken down by country and singleton status.


In [ ]:
reload_pipeline()
MAX_STACK_ENTITIES = 60_000
free_memory()
ensemble.run(ROOT, DP / "features_train.parquet", val_frac=0.2, seed=42,
             laya_batch_size=ensemble.LAYA_BATCH_SIZE, laya_top_n=ensemble.LAYA_TOP_N,
             laya_min_gbdt_prob=ensemble.LAYA_MIN_GBDT_PROB, max_entities=MAX_STACK_ENTITIES)
free_memory()


In [ ]:
ens_report = json.load(open(DP / "ensemble_val_report.json"))
scores = {name: ens_report[name]["threshold_sweep"]["best"] for name in ("gbdt_only_baseline", "logistic", "gbdt_alt")}
for name, best in scores.items():
    print(f"{name:20s} threshold={best['threshold']:<5} macro_F0.5={best['macro_f0_5']:.4f} "
          f"precision={best['precision']:.4f} recall={best['recall']:.4f}")

best_name = max(scores, key=lambda n: scores[n]["macro_f0_5"])
BEST_MODEL = "gbdt_only" if best_name == "gbdt_only_baseline" else best_name
print(f"\nUsing for the test set: {BEST_MODEL}"
      + ("  (Laya didn't beat GBDT alone here, so Section 7 skips Laya)" if BEST_MODEL == "gbdt_only" else ""))
print(json.dumps(ens_report[best_name]["breakdown"], indent=2))


## 7. Full test-set inference

Blocks **every** test S1 entity, then runs features, GBDT, Laya (unless `BEST_MODEL` is
`gbdt_only`) and the ensemble with its threshold. Writes `output/candidate_pairs.tsv` and
`output/matching_results.tsv`.


In [ ]:
reload_pipeline()
free_memory()
predict.run(ROOT, k=30, stack_model=BEST_MODEL, laya_batch_size=ensemble.LAYA_BATCH_SIZE,
            threshold_override=None)
free_memory()


## 8. Validate before submitting

The challenge's own validator, run from `student_resource/`.


In [ ]:
%cd $STUDENT_RESOURCE_DIR
!python3 utils/validate_submission.py --matching output/matching_results.tsv --candidate output/candidate_pairs.tsv --test-dir dataset/test
%cd $PIPELINE_DIR


## 9. Package the submission zip

`output/`, `code/business_entity_resolution/` (src, README, requirements) and
`Documentation_template.md`, in the layout the challenge requires.


In [ ]:
zip_root = f"{WORKDIR}/submission_package"
if os.path.isdir(zip_root):
    shutil.rmtree(zip_root)
os.makedirs(zip_root)
shutil.copytree(f"{STUDENT_RESOURCE_DIR}/output", f"{zip_root}/output")
shutil.copytree(PIPELINE_DIR, f"{zip_root}/code/business_entity_resolution",
                ignore=shutil.ignore_patterns("__pycache__"))
shutil.copy(f"{STUDENT_RESOURCE_DIR}/Documentation_template.md", f"{zip_root}/Documentation_template.md")
archive_path = shutil.make_archive(f"{WORKDIR}/submission", "zip", zip_root)
print("Wrote", archive_path, "(Kaggle: Output pane after saving a version)")
